# Limpieza de Datos – Marine (Versión Corregida)

## 1. Importación de paqueterías 


In [1]:
import os
import pandas as pd
from tabulate import tabulate
from unidecode import unidecode
import re
import numpy as np
from rapidfuzz import fuzz, process
from collections import defaultdict, Counter
import ahocorasick
import pandas as pd
import re
from unidecode import unidecode
from collections import OrderedDict
# from opcion1_estado_offline import agregar_estado
# o: from opcion2_estado_geocoding import agregar_estado_hibrido


## 1.5 Importación de configuración centralizada

In [2]:
# =============================================================================
# Importación centralizada de configuración
# =============================================================================
import sys
sys.path.insert(0, r'C:/Users/IKAL14/Documents/Integral/Marine/Diccionarios')
from config_marine import *


## 2. Carga de datos

In [3]:
#Carga de archivo original
# RECOMENDADO
import os
BASE_DIR = DIRECTORIO_PROYECTO  # Desde config_marine.py
PERIODO = os.environ.get("PERIODO", "202512")
archivo_original = f"{BASE_DIR}/Procesados/{PERIODO}/Final/{PERIODO}_Siniestros_Marine.xlsx"

# Cargar archivo Excel o CSV
df = pd.read_excel(archivo_original, dtype={"INWARD POLICY N°": str})   # o pd.read_csv("datos.csv")
#archivoreadme= "C:\Users\IKAL14\OneDrive - Kot Insurance Company AG\Vida\Vida 2025\readme.txt"

### Extracción del nombre del archivo


In [4]:
# Extraer año y mes del nombre del archivo
nombre_base = os.path.basename(archivo_original)  # '202508_Siniestros_Vida.xlsx'
ano_mes = nombre_base[:6]                        # '202508'
ano = ano_mes[:4]                                # '2025'
mes = ano_mes[4:]                                # '08'
mes_num = int(ano_mes[4:])                        # 8 (convertido a entero)

nombre_archivo_salida = f"{ano_mes}_Siniestros_Marine_PROCESADO.xlsx"

# Carpeta y nombre de archivo de salida
carpeta_salida = f"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/06-RM - 02-Actuarial/06. DATABASE/03 Transporte, Carga & Embarcaciones/02 Bases Actuarial/{ano_mes}"
carpeta_salida2 = f"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{ano}/{ano_mes}"

# Crear carpeta si no existe
os.makedirs(carpeta_salida, exist_ok=True)
os.makedirs(carpeta_salida2, exist_ok=True)

# Quitar filas completamente vacías
df = df.dropna(how="all")

#Agregar columna con mes de carga
df["MES CARGA"] = pd.Timestamp(f"{ano}-{mes}")



## Diccionarios

In [5]:
# ================================
# CONFIGURACIÓN DE MAPEOS (Siempre en MAYÚSCULAS)
# ================================
cover_map = COVER_MAP  # Desde config_marine.py

# maplobinward: COVER canónico → LoB-Inward final (igual a NB1 cell 24)
maplobinward = MAP_LOB_INWARD  # Desde config_marine.py


## 3. Limpieza de datos

In [6]:
# =============================================================================
# 3. TRATAMIENTO DE NULOS EN LOB-INWARD, ALERTAS DE COVER Y REEMPLAZOS
# =============================================================================

# Definición global de cadenas que representan valores vacíos o inválidos
# VALORES_VACIOS -> importado de config_marine.py

# --- PASO 3a: Identificación de registros vacíos en LoB-Inward y COVER ---
mask_lob_vacio = (
    df["LoB-Inward"].isna() | 
    df["LoB-Inward"].astype(str).str.strip().str.upper().isin(VALORES_VACIOS)
)

mask_cover_vacio = (
    df["COVER"].isna() | 
    df["COVER"].astype(str).str.strip().str.upper().isin(VALORES_VACIOS)
)

# --- PASO 3b: Sistema de Alertas para la columna COVER ---
# Si se detecta que existen coberturas vacías, enviamos un aviso preventivo a consola
if mask_cover_vacio.any():
    print("=" * 70)
    print(f"⚠️ ALERT: Se detectaron {mask_cover_vacio.sum()} registros con la columna 'COVER' vacía o inválida.")
    print("Revisa la fuente si estos registros no logran mapear correctamente a LoB-Inward.")
    print("=" * 70)

# --- PASO 3c: Mapeo de LoB-Inward a partir de COVER ---
# Rellenamos LoB-Inward únicamente si está vacío Y si el campo COVER sí contiene datos válidos para mapear
mask_proceder_mapeo = mask_lob_vacio & ~mask_cover_vacio
if mask_proceder_mapeo.any():
    df.loc[mask_proceder_mapeo, "LoB-Inward"] = (
        df.loc[mask_proceder_mapeo, "COVER"].map(maplobinward)
    )
    print(f"Asignación automática: Se completó 'LoB-Inward' para {mask_proceder_mapeo.sum()} registros usando 'COVER'.")

# =============================================================================
# 3b. HERENCIA DE LOB-INWARD Y COVER POR SINIESTRO (CLAIM NUMBER)
# =============================================================================
# Mantiene la consistencia propagando valores válidos entre filas del mismo siniestro
if "CLAIM NUMBER" in df.columns:
    VACIOS = VALORES_VACIOS
    def primer_valor_valido(s, vacios):
        validos = s[s.notna() & ~s.astype(str).str.strip().str.upper().isin(vacios)]
        return validos.iloc[0] if not validos.empty else s.iloc[0]

    # Heredar LoB-Inward
    mask_lob_vacio = (
        df["LoB-Inward"].isna() |
        df["LoB-Inward"].astype(str).str.strip().str.upper().isin(VACIOS)
    )
    lob_heredado = df.groupby("CLAIM NUMBER")["LoB-Inward"].transform(
        lambda s: primer_valor_valido(s, VACIOS)
    )
    df.loc[mask_lob_vacio, "LoB-Inward"] = lob_heredado[mask_lob_vacio]

    # Heredar COVER
    mask_cover_vacio = (
        df["COVER"].isna() |
        df["COVER"].astype(str).str.strip().str.upper().isin(VACIOS)
    )
    cover_heredado = df.groupby("CLAIM NUMBER")["COVER"].transform(
        lambda s: primer_valor_valido(s, VACIOS)
    )
    df.loc[mask_cover_vacio, "COVER"] = cover_heredado[mask_cover_vacio]

    # Heredar fechas de poliza
    for col_fecha in ["POLICY PERIOD START DATE", "POLICY PERIOD END DATE"]:
        if col_fecha in df.columns:
            fecha_heredada = df.groupby("CLAIM NUMBER")[col_fecha].transform("first")
            df[col_fecha] = df[col_fecha].fillna(fecha_heredada)
    print('LoB-Inward/COVER heredados entre filas del mismo siniestro.')

# =============================================================================
# 3c. RECONSTRUCCIÓN DE IDENTIFICADORES (CLAIMS-ID & KEY LOB)
# =============================================================================

# --- PASO 3c.1: Reconstrucción de CLAIMS-ID ---
# Identificamos valores nulos o que contengan explícitamente la leyenda 'NO ESPECIFICADO'
mask_claims_id_malo = (
    df["CLAIMS-ID"].isna() |
    df["CLAIMS-ID"].astype(str).str.strip().str.upper().str.contains('NO ESPECIFICADO', na=False)
)

# Si hay llaves dañadas y contamos con las columnas origen necesarias, las regeneramos
if mask_claims_id_malo.any() and "LoB-Inward" in df.columns and "CLAIM NUMBER" in df.columns:
    df.loc[mask_claims_id_malo, "CLAIMS-ID"] = (
        df.loc[mask_claims_id_malo, "CLAIM NUMBER"].astype(str)
        + "-"
        + df.loc[mask_claims_id_malo, "LoB-Inward"].astype(str)
    )
    print(f'⚙️ CLAIMS-ID reconstruido de forma compuesta para {mask_claims_id_malo.sum()} registros.')

# --- PASO 3c.2: Reconstrucción de KEY LOB ---
# Identificamos valores nulos o que contengan explícitamente la leyenda 'NO ESPECIFICADO'
if "KEY LOB" in df.columns:
    mask_keylob_malo = (
        df["KEY LOB"].isna() |
        df["KEY LOB"].astype(str).str.strip().str.upper().str.contains('NO ESPECIFICADO', na=False)
    )
    
    if mask_keylob_malo.any() and "LoB-Inward" in df.columns and "CLAIM NUMBER" in df.columns:
        df.loc[mask_keylob_malo, "KEY LOB"] = (
            df.loc[mask_keylob_malo, "CLAIM NUMBER"].astype(str)
            + "-"
            + df.loc[mask_keylob_malo, "LoB-Inward"].astype(str)
        )
        print(f'⚙️ KEY LOB reconstruido de forma compuesta para {mask_keylob_malo.sum()} registros.')

⚠️ ALERT: Se detectaron 2 registros con la columna 'COVER' vacía o inválida.
Revisa la fuente si estos registros no logran mapear correctamente a LoB-Inward.
Asignación automática: Se completó 'LoB-Inward' para 977 registros usando 'COVER'.
LoB-Inward/COVER heredados entre filas del mismo siniestro.
⚙️ CLAIMS-ID reconstruido de forma compuesta para 979 registros.
⚙️ KEY LOB reconstruido de forma compuesta para 2 registros.


In [7]:

# ================================
# LIMPIEZA POR COLUMNAS
# ================================

# 1. INWARD POLICY N°
df["INWARD POLICY N°"] = (
    df["INWARD POLICY N°"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)  # Quita .0 de floats
    .str.strip("'")  # Quita comillas previas
)

# 1b. CLAIM NUMBER — quitar comilla de Excel y limpiar derivados
for col in ["CLAIM NUMBER", "CLAIM NUMBER ORIGINAL BDX"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lstrip("'")
            .str.strip()
            .replace({'nan': '', 'None': ''})
        )

# 1c. SUBSIDIARY — quitar comilla de Excel
if "SUBSIDIARY" in df.columns:
    df["SUBSIDIARY"] = (
        df["SUBSIDIARY"]
        .astype(str)
        .str.strip()
        .str.lstrip("'")
        .str.strip()
    )

# 2. DESCRIPTION (Unificado y optimizado sin .apply)
df["DESCRIPTION"] = (
    df["DESCRIPTION"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(
        r"['\"\[\]\(\)\{\}•*“”]", "", regex=True
    )  # Remueve símbolos respetando Ñ, acentos y guiones
)

# 3. COVER
df["COVER"] = (
    df["COVER"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.replace(
        "[\xa0\u200b\ufeff\t\r]", " ", regex=True
    )  # Caracteres invisibles a espacio
    .str.replace(r"\s+", " ", regex=True)  # Colapsa múltiples espacios
    .str.strip()
    .str.upper()
    .replace(cover_map)  # Mapeo homologado
)


In [8]:

# 4. LoB-Inward (Tratamiento de nulos + Herencia + Reemplazos)
#Rellenar lobinward vacios según el maplobinward y el COVER canónico
mask = (
    df["LoB-Inward"].isna()
    | df["LoB-Inward"].astype(str).str.strip().eq("")
    | df["LoB-Inward"].astype(str).str.upper().str.strip().eq("NO ESPECIFICADO")
    | df["LoB-Inward"].astype(str).str.upper().str.strip().eq("NAN")
)

df.loc[mask, "LoB-Inward"] = (
    df.loc[mask, "COVER"].map(maplobinward)
)

# 4b. Heredar LoB-Inward y COVER entre filas del mismo CLAIM NUMBER
# (cuando un siniestro tiene 2 filas: una con LoB vacío y otra con LoB válido)
if "CLAIM NUMBER" in df.columns:
    VACIOS = ['', 'nan', 'NAN', 'NO ESPECIFICADO']
    def primer_valor_valido(s, vacios):
        validos = s[s.notna() & ~s.astype(str).str.strip().str.upper().isin(vacios)]
        return validos.iloc[0] if not validos.empty else s.iloc[0]

    # Heredar LoB-Inward
    mask_lob_vacio = (
        df["LoB-Inward"].isna() |
        df["LoB-Inward"].astype(str).str.strip().str.upper().isin(VACIOS)
    )
    lob_heredado = df.groupby("CLAIM NUMBER")["LoB-Inward"].transform(
        lambda s: primer_valor_valido(s, VACIOS)
    )
    df.loc[mask_lob_vacio, "LoB-Inward"] = lob_heredado[mask_lob_vacio]

    # Heredar COVER
    mask_cover_vacio = (
        df["COVER"].isna() |
        df["COVER"].astype(str).str.strip().str.upper().isin(VACIOS)
    )
    cover_heredado = df.groupby("CLAIM NUMBER")["COVER"].transform(
        lambda s: primer_valor_valido(s, VACIOS)
    )
    df.loc[mask_cover_vacio, "COVER"] = cover_heredado[mask_cover_vacio]

    # Heredar fechas de poliza
    for col_fecha in ["POLICY PERIOD START DATE", "POLICY PERIOD END DATE"]:
        if col_fecha in df.columns:
            fecha_heredada = df.groupby("CLAIM NUMBER")[col_fecha].transform("first")
            df[col_fecha] = df[col_fecha].fillna(fecha_heredada)
    print(f'LoB-Inward/COVER heredados entre filas del mismo siniestro')

# 4d. Si despues de todo el proceso LoB-Inward o COVER siguen vacios -> NO ESPECIFICADO
VACIOS_FINAL = ['', 'nan', 'NAN', 'None', 'NONE']

mask_lob_final = (
    df['LoB-Inward'].isna() |
    df['LoB-Inward'].astype(str).str.strip().isin(VACIOS_FINAL)
)
mask_cover_final = (
    df['COVER'].isna() |
    df['COVER'].astype(str).str.strip().isin(VACIOS_FINAL)
)

# Solo aplica cuando AMBOS estan vacios (no hay nada de donde inferir)
mask_ambos_vacios = mask_lob_final & mask_cover_final
if mask_ambos_vacios.any():
    df.loc[mask_ambos_vacios, 'LoB-Inward'] = 'NO ESPECIFICADO'
    df.loc[mask_ambos_vacios, 'COVER']      = 'NO ESPECIFICADO'
    print(f'Registros con LoB-Inward y COVER vacios marcados como NO ESPECIFICADO: {mask_ambos_vacios.sum()}')
    if 'CLAIM NUMBER' in df.columns:
        print(df.loc[mask_ambos_vacios, ['CLAIM NUMBER', 'LoB-Inward', 'COVER']].to_string(index=False))

# Si solo LoB-Inward sigue vacio (pero COVER tiene valor), marcar LoB como NO ESPECIFICADO
mask_solo_lob = mask_lob_final & ~mask_cover_final
if mask_solo_lob.any():
    df.loc[mask_solo_lob, 'LoB-Inward'] = 'NO ESPECIFICADO'
    print(f'Registros con solo LoB-Inward vacio (COVER existe) marcados como NO ESPECIFICADO: {mask_solo_lob.sum()}')

# 4c. Reconstruir CLAIMS-ID y KEY LOB donde quedaron con 'NO ESPECIFICADO'
mask_claims_id_malo = (
    df["CLAIMS-ID"].astype(str).str.contains('NO ESPECIFICADO', na=False) |
    df["CLAIMS-ID"].isna()
)
if mask_claims_id_malo.any() and "LoB-Inward" in df.columns:
    df.loc[mask_claims_id_malo, "CLAIMS-ID"] = (
        df.loc[mask_claims_id_malo, "CLAIM NUMBER"].astype(str)
        + "-"
        + df.loc[mask_claims_id_malo, "LoB-Inward"].astype(str)
    )
    print(f'CLAIMS-ID reconstruido para {mask_claims_id_malo.sum()} registros')

if "KEY LOB" in df.columns:
    mask_keylob_malo = (
        df["KEY LOB"].astype(str).str.contains('NO ESPECIFICADO', na=False) |
        df["KEY LOB"].isna()
    )
    if mask_keylob_malo.any():
        df.loc[mask_keylob_malo, "KEY LOB"] = (
            df.loc[mask_keylob_malo, "CLAIM NUMBER"].astype(str)
            + "-"
            + df.loc[mask_keylob_malo, "LoB-Inward"].astype(str)
        )
        print(f'KEY LOB reconstruido para {mask_keylob_malo.sum()} registros')


LoB-Inward/COVER heredados entre filas del mismo siniestro
Registros con solo LoB-Inward vacio (COVER existe) marcados como NO ESPECIFICADO: 2
CLAIMS-ID reconstruido para 2 registros
KEY LOB reconstruido para 2 registros


In [9]:

# 5. OTRAS COLUMNAS (Rellenos fijos de nulos)
df["COVERAGE"] = df["COVERAGE"].fillna("NO ESPECIFICADO")

#Normalizar LOCATION (Unificado y optimizado sin .apply)
def normalizar(texto):
    """Normaliza texto: mayusculas, sin acentos, sin puntuacion ruidosa."""
    if pd.isna(texto) or str(texto).strip() in ("", "-", "nan", "N/A"):
        return ""
    t = unidecode(str(texto)).upper().strip()
    t = re.sub(r"[;:'\"\[\]\(\)\{\}*#!]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["LOCATION"] = df["LOCATION"].fillna("NO ESPECIFICADO")
df.loc[df["LOCATION"] == "-", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Pendiente", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "pendiente", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "PENDIENTE", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "CD", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Litigio", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "MANCHAS A UN BUQUE LPG BERING GAS", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Terminal de Pemex", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "REF.RC-R-204-2010", "LOCATION"] = "NO ESPECIFICADO"
df.loc[df["LOCATION"] == "Lugar/Terminal: PACIFIC AREA LIGHTERING", "LOCATION"] = "PACIFICO NORTE"

#Apply normalización a toda la columna LOCATION
df["LOCATION"] = df["LOCATION"].apply(normalizar)

#Rellenar CLAIMS-ID faltantes con combinación de CLAIM NUMBER + COVER (si ambos existen)
mask = (
    df["CLAIMS-ID"].isna() &
    df["CLAIM NUMBER"].notna() &
    df["COVER"].notna()
)

df.loc[mask, "CLAIMS-ID"] = (
    df.loc[mask, "CLAIM NUMBER"].astype(str)
    + "-"
    + df.loc[mask, "COVER"].astype(str)
)

#Rellenar CLAIM NUMBER faltantes con CLAIMS-ID si tiene formato esperado (ej. "12345-CASCO Y MAQ.")
mask = (
    df["CLAIM NUMBER"].isna() &
    df["CLAIMS-ID"].notna() &
    df["COVER"].notna()
)

df.loc[mask, "CLAIM NUMBER"] = (
    df.loc[mask, "CLAIMS-ID"].astype(str).str.split("-").str[0]
)

# Eliminación de columnas sobrantes
df = df.drop(["NOTAS", "REF. PEMEX"], axis=1, errors="ignore")

# ================================
# NORMALIZACION Y VALIDACION DE STATUS
# ================================

VALID_STATUSES = VALID_STATUSES  # Desde config_marine.py (ya importado)

# Paso 1: Guardar valor original y normalizar tipografía
df["STATUS_ORIGINAL"] = df["STATUS"].copy()

df["STATUS"] = (
    df["STATUS"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"NAN": "", "NONE": "", "N/A": "", "NA": "", "-": "",
              "--": "", ".": "", "#N/A": "", "NULL": "", "NULO": ""})
    .replace(r"^\s*$", "", regex=True)
)

# Paso 2: Detectar vacios e invalidos (ambos son candidatos al rescue)
mask_vacio    = df["STATUS"].isin(["", "NO ESPECIFICADO"])
mask_invalido = ~df["STATUS"].isin(VALID_STATUSES) & ~mask_vacio
sin_status    = df[mask_vacio | mask_invalido].copy()

# ================================
# Paso 3: Rescue de STATUS faltante
# Fuente 1: archivo original del periodo (BDX crudo)
# Fuente 2: base procesada mas reciente (hasta 12 meses atras)
# ================================

def _buscar_status_en_fuente(claims_sin_status, df_fuente, fuente_label):
    recuperados = {}
    if df_fuente is None or df_fuente.empty:
        print("  Fuente no disponible:", fuente_label)
        return recuperados
    if "STATUS" not in df_fuente.columns:
        print("  Sin columna STATUS en:", fuente_label)
        return recuperados
    df_f = df_fuente.copy()
    df_f["STATUS"] = (
        df_f["STATUS"]
        .astype(str).str.strip().str.upper()
        .replace({"NAN": "", "NONE": "", "N/A": "", "NA": "", "-": ""})
        .replace(r"^\s*$", "", regex=True)
    )
    df_v = df_f[df_f["STATUS"].isin(VALID_STATUSES)].copy()
    if df_v.empty:
        print("  Sin STATUS validos en:", fuente_label)
        return recuperados
    if "CLAIM NUMBER" in df_v.columns:
        df_v["_CN_NORM"] = (
            df_v["CLAIM NUMBER"].astype(str)
            .str.replace(r"\s+", "", regex=True).str.upper().str.strip()
        )
    if "CLAIMS-ID" in df_v.columns:
        df_v["_CID_NORM"] = (
            df_v["CLAIMS-ID"].astype(str)
            .str.replace(r"\s+", "", regex=True).str.upper().str.strip()
        )
    for idx_row, row in claims_sin_status.iterrows():
        if "CLAIM NUMBER" in df_v.columns and "_CN_NORM" in df_v.columns:
            val_raw  = str(row.get("CLAIM NUMBER", "")).strip()
            val_norm = val_raw.replace(" ", "").upper()
            if val_raw and val_raw.lower() not in ("nan", "none", ""):
                m = df_v[df_v["CLAIM NUMBER"].astype(str).str.strip() == val_raw]
                if m.empty:
                    m = df_v[df_v["_CN_NORM"] == val_norm]
                if not m.empty:
                    recuperados[idx_row] = m.iloc[0]["STATUS"]
                    continue
        if "CLAIMS-ID" in df_v.columns and "_CID_NORM" in df_v.columns:
            val_raw  = str(row.get("CLAIMS-ID", "")).strip()
            val_norm = val_raw.replace(" ", "").upper()
            if val_raw and val_raw.lower() not in ("nan", "none", ""):
                m = df_v[df_v["CLAIMS-ID"].astype(str).str.strip() == val_raw]
                if m.empty:
                    m = df_v[df_v["_CID_NORM"] == val_norm]
                if not m.empty:
                    recuperados[idx_row] = m.iloc[0]["STATUS"]
    print("  Recuperados de", fuente_label + ":", len(recuperados))
    return recuperados


def _buscar_archivo_anterior(periodo_actual, base_dir):
    ruta_onedrive = r"C:\Users\IKAL14\OneDrive - Kot Insurance Company AG\Transporte, Carga y Embarcaciones"
    _ano = int(periodo_actual[:4])
    _mes = int(periodo_actual[4:])
    for _ in range(12):
        if _mes == 1:
            _mes, _ano = 12, _ano - 1
        else:
            _mes -= 1
        p = str(_ano) + str(_mes).zfill(2)
        candidatos = [
            os.path.join(ruta_onedrive, str(_ano), p, p + "_Siniestros_Marine_PROCESADO.xlsx"),
            os.path.join(base_dir, "Procesados", p, "Final", p + "_Siniestros_Marine_PROCESADO.xlsx"),
            os.path.join(base_dir, "Procesados", p, "Final", p + "_Siniestros_Marine.xlsx"),
            os.path.join(os.path.dirname(base_dir), p, p + "_Siniestros_Marine_PROCESADO.xlsx"),
        ]
        for ruta in candidatos:
            if os.path.exists(ruta):
                return ruta, p
    return None, None


if not sin_status.empty:
    print("Registros sin STATUS valido:", len(sin_status),
          "(vacios:", int(mask_vacio.sum()), "| invalidos:", int(mask_invalido.sum()), ")")
    print("Iniciando rescue en fuentes externas...")
    recuperados = {}

    try:
        df_f1 = pd.read_excel(archivo_original, dtype=str)
        recuperados.update(
            _buscar_status_en_fuente(sin_status, df_f1, "Archivo original " + PERIODO)
        )
        del df_f1
    except Exception as e:
        print("  Error leyendo archivo original:", e)

    aun_sin = sin_status[~sin_status.index.isin(recuperados.keys())]
    if not aun_sin.empty:
        ruta_ant, periodo_ant = _buscar_archivo_anterior(PERIODO, BASE_DIR)
        if ruta_ant is None:
            print("  No se encontro base anterior (revisados 12 meses atras)")
        else:
            print("  Usando base anterior:", periodo_ant, "->", ruta_ant)
            try:
                df_f2 = pd.read_excel(ruta_ant, dtype=str)
                recuperados.update(
                    _buscar_status_en_fuente(aun_sin, df_f2, "Base procesada " + periodo_ant)
                )
                del df_f2
            except Exception as e:
                print("  Error leyendo base anterior:", e)

    for idx_rec, status_rec in recuperados.items():
        df.at[idx_rec, "STATUS"] = status_rec

    print("Total STATUS recuperados:", len(recuperados))
    print()

    mask_vacio    = df["STATUS"].isin(["", "NO ESPECIFICADO"])
    mask_invalido = ~df["STATUS"].isin(VALID_STATUSES) & ~mask_vacio

# ================================
# Paso 4: Si aun quedan sin STATUS -> reporte + detener ejecucion
# ================================
problemas = df[mask_vacio | mask_invalido].copy()

if not problemas.empty:
    n_vacios    = int(mask_vacio.sum())
    n_invalidos = int(mask_invalido.sum())
    print("=" * 65)
    print("ERROR DE VALIDACION - STATUS INCOMPLETO O INVALIDO")
    print("=" * 65)
    print("Registros sin STATUS (vacio):", n_vacios)
    print("Registros con STATUS invalido:", n_invalidos)
    print("Valores validos:", VALID_STATUSES)
    print()
    cols_reporte = [c for c in ["CLAIM NUMBER", "CLAIMS-ID", "INWARD POLICY N°",
                                "COVER", "STATUS_ORIGINAL", "STATUS",
                                "Cumulative CLAIMS PAID", "OSLR Inward"]
                    if c in problemas.columns]
    print(problemas[cols_reporte].to_string(index=False))
    print()
    print("Accion requerida: corrige el STATUS en el BDX y re-ejecuta.")
    print("=" * 65)
    raise ValueError(
        "Se encontraron " + str(len(problemas)) + " registros con STATUS invalido o vacio "
        "que no pudieron resolverse. Corrige el archivo fuente."
    )

# ================================
# Paso 5: Validacion de coherencia financiera por STATUS
# APLICA UNICAMENTE a polizas NCGL-070
# Reglas:
#   C -> CLAIMS == 0  y  OSLR == 0
#   T -> OSLR > 0
#   P -> CLAIMS > 0   y  OSLR == 0
# ================================
df["NOTA_VALIDACION"] = ""          # or pd.NA, or some default value
df["NOTA_VALIDACION"] = df["NOTA_VALIDACION"].astype("string")

COL_EXPENSES  = "Cumulative EXPENSES PAID"
COL_VALUATION = "Cumulative VALUATION EXPENSES"
COL_CLAIMS    = "Cumulative CLAIMS PAID"
COL_OSLR      = "OSLR Inward"
COL_POLICY    = "INWARD POLICY N°"
cols_fin      = [COL_EXPENSES, COL_VALUATION, COL_CLAIMS, COL_OSLR]

# Inicializar columna de notas si no existe
if "NOTA_VALIDACION" not in df.columns:
    df["NOTA_VALIDACION"] = ""

if not all(c in df.columns for c in cols_fin):
    print("Advertencia: faltan columnas financieras:",
          [c for c in cols_fin if c not in df.columns])
elif COL_POLICY not in df.columns:
    print("Advertencia: columna", COL_POLICY, "no encontrada - omitiendo validacion financiera.")
else:
    mask_ncgl = df[COL_POLICY].astype(str).str.strip().str.upper().str.startswith("NCGL-070")
    df_ncgl   = df[mask_ncgl].copy()

    if df_ncgl.empty:
        print("Advertencia: no se encontraron registros de polizas NCGL-070.")
    else:
        print("Registros sujetos a validacion financiera (NCGL-070):", len(df_ncgl))

        _e  = pd.to_numeric(df_ncgl[COL_EXPENSES],  errors="coerce").fillna(0)
        _v  = pd.to_numeric(df_ncgl[COL_VALUATION], errors="coerce").fillna(0)
        _cl = pd.to_numeric(df_ncgl[COL_CLAIMS],    errors="coerce").fillna(0)
        _os = pd.to_numeric(df_ncgl[COL_OSLR],      errors="coerce").fillna(0)

        # Construir etiqueta de violacion por registro
        notas_viol = pd.Series("", index=df_ncgl.index)
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "C") & (_cl != 0)),
            "STATUS C pero CLAIMS != 0"
        )
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "C") & (_os != 0) & (notas_viol == "")),
            "STATUS C pero OSLR != 0"
        )
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "C") & (_cl != 0) & (_os != 0) & (notas_viol != "STATUS C pero CLAIMS != 0")),
            "STATUS C pero OSLR != 0"
        )
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "T") & (_os <= 0)),
            "STATUS T pero OSLR <= 0"
        )
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "P") & (_cl <= 0)),
            "STATUS P pero CLAIMS <= 0"
        )
        notas_viol = notas_viol.where(
            ~((df_ncgl["STATUS"] == "P") & (notas_viol == "")),
            "STATUS P pero OSLR != 0"
        )

        viol_C = (df_ncgl["STATUS"] == "C") & ((_cl != 0) | (_os != 0))
        viol_T = (df_ncgl["STATUS"] == "T") & (_os <= 0)
        viol_P = (df_ncgl["STATUS"] == "P") & ((_cl <= 0))
        mask_viol = viol_C | viol_T | viol_P

        violaciones_fin = df_ncgl[mask_viol].copy()
        violaciones_fin["MOTIVO_VIOLACION"] = notas_viol[mask_viol]

        if not violaciones_fin.empty:
            print("=" * 65)
            print("ALERTA - INCOHERENCIA STATUS vs. FINANZAS (NCGL-070)")
            print("=" * 65)
            print("Total registros con incoherencia:", len(violaciones_fin))
            print()
            print("Reglas: C->CLAIMS==0,OSLR==0 | T->OSLR>0 | P->CLAIMS>0,OSLR==0")
            print()
            cols_rep = [c for c in ["CLAIM NUMBER", "CLAIMS-ID", COL_POLICY, "COVER",
                                    "STATUS", COL_CLAIMS, COL_OSLR, "MOTIVO_VIOLACION"]
                        if c in violaciones_fin.columns]
            print(violaciones_fin[cols_rep].to_string(index=False))
            print()
            print("=" * 65)
            print("Opciones:")
            print("  [1] Ya corregí el BDX — re-valida automáticamente")
            print("  [2] Continuar y agregar nota a los registros problemáticos")
            print("  [3] Detener ejecución")
            print("=" * 65)

            respuesta = input("¿Qué deseas hacer? (1/2/3): ").strip()

            if respuesta == "1":
                # Releer el archivo y reintentar (el usuario corrigió el BDX)
                print("Releyendo archivo original para revalidar...")
                df_reload = pd.read_excel(archivo_original, dtype={"INWARD POLICY N": str})
                df_reload = df_reload.dropna(how="all")
                # Actualizar CLAIMS y OSLR en df para los índices con violación
                for col in [COL_CLAIMS, COL_OSLR, COL_EXPENSES, COL_VALUATION, "STATUS"]:
                    if col in df_reload.columns and col in df.columns:
                        df.loc[violaciones_fin.index, col] = df_reload.loc[
                            df_reload.index.isin(violaciones_fin.index), col
                        ].values if len(df_reload) == len(df) else df.loc[violaciones_fin.index, col]
                print("Re-validacion completada. Continua la ejecucion.")
                print("NOTA: Si la violacion persiste se detectara en el siguiente paso.")

            elif respuesta == "2":
                nota_usuario = input("Escribe la nota a agregar (Enter para usar el motivo automatico): ").strip()
                for idx_v in violaciones_fin.index:
                   nota_final = (
                       nota_usuario
                       if nota_usuario
                       else violaciones_fin.loc[idx_v, "MOTIVO_VIOLACION"]
                    )
                   df.at[idx_v, "NOTA_VALIDACION"] = nota_final
                print("Notas agregadas a", len(violaciones_fin), "registros. Continuando ejecucion.")
            else:
                raise ValueError(
                    "Ejecucion detenida por el usuario. Se encontraron " +
                    str(len(violaciones_fin)) + " registros NCGL-070 con STATUS "
                    "financieramente incoherente. Corrige el archivo fuente y re-ejecuta."
                )

# Paso 6: Confirmacion final
print("Validacion de STATUS completada sin errores.")
print()
print("Distribucion de STATUS:")
print(df["STATUS"].value_counts().to_string())
if df["NOTA_VALIDACION"].ne("").any():
    print()
    print("Registros con NOTA_VALIDACION:", df["NOTA_VALIDACION"].ne("").sum())
# ================================

# ============================================================
# VALIDACIÓN: STATUS 'P' con OSLR > 0
# Si STATUS es P (pagado/cerrado), el OSLR debe ser 0.
# Excepción: si NOTA_VALIDACION ya tiene justificación,
# se permite OSLR > 0 pero se registra la nota.
# ============================================================

if 'NOTA_VALIDACION' not in df.columns:
    df['NOTA_VALIDACION'] = ''
df['NOTA_VALIDACION'] = df['NOTA_VALIDACION'].fillna('')

mask_legacy = df['INWARD POLICY N°'].isin(LIST_LEGACY)

mask_p_oslr = (
    df['STATUS'].astype(str).str.strip() == 'P'
) & (df['OSLR Inward'] > 0) & (~mask_legacy)

mask_sin_nota = mask_p_oslr & (df['NOTA_VALIDACION'].str.strip() == '')
mask_con_nota = mask_p_oslr & (df['NOTA_VALIDACION'].str.strip() != '')

# Sin justificación → zerear OSLR y registrar nota
if mask_sin_nota.any():
    df.loc[mask_sin_nota, 'OSLR Inward'] = 0
    df.loc[mask_sin_nota, 'NOTA_VALIDACION'] = 'STATUS P: OSLR zerado por ausencia de justificación'
    print(f"⚠️  {mask_sin_nota.sum()} registro(s) con STATUS=P y OSLR>0 zerados:")
    print(df.loc[mask_sin_nota, ['CLAIM NUMBER', 'STATUS', 'NOTA_VALIDACION']].to_string())
else:
    print("✅ No hay registros con STATUS=P y OSLR>0 sin justificación")

# Con justificación → OSLR se conserva, solo se reporta
if mask_con_nota.any():
    print(f"\nℹ️  {mask_con_nota.sum()} registro(s) con STATUS=P y OSLR>0 permitido por NOTA_VALIDACION:")
    print(df.loc[mask_con_nota, ['CLAIM NUMBER', 'STATUS', 'OSLR Inward', 'NOTA_VALIDACION']].to_string())


Registros sin STATUS valido: 32 (vacios: 0 | invalidos: 32 )
Iniciando rescue en fuentes externas...
  Recuperados de Archivo original 202512: 8
  Usando base anterior: 202510 -> C:\Users\IKAL14\OneDrive - Kot Insurance Company AG\Transporte, Carga y Embarcaciones\2025\202510\202510_Siniestros_Marine_PROCESADO.xlsx
  Recuperados de Base procesada 202510: 24
Total STATUS recuperados: 32

Registros sujetos a validacion financiera (NCGL-070): 786
ALERTA - INCOHERENCIA STATUS vs. FINANZAS (NCGL-070)
Total registros con incoherencia: 7

Reglas: C->CLAIMS==0,OSLR==0 | T->OSLR>0 | P->CLAIMS>0,OSLR==0

CLAIM NUMBER         CLAIMS-ID INWARD POLICY N°             COVER STATUS  Cumulative CLAIMS PAID  OSLR Inward          MOTIVO_VIOLACION
 105280/2025   105280/2025-P&I NCGL-070-1002258               P&I      T                     0.0          0.0   STATUS T pero OSLR <= 0
 106122/2018   106122/2018-P&I NCGL-070-1000773               P&I      T                     0.0          0.0   STATUS T pero 

In [10]:
print(f"Registros después de limpieza básica: {len(df)}")

Registros después de limpieza básica: 4294


### 3.2 Normalización para Location


In [11]:
import re
import unicodedata
from collections import OrderedDict
from typing import Optional
import pandas as pd
from unidecode import unidecode
from rapidfuzz import fuzz, process

# ════════════════════════════════════════════════════════════════════════════
# CLASIFICADOR DE LOCATIONS  v3.0  (fusión location_classifier + Marine)
# Identifica ESTADO y PAIS a partir de texto libre en columna LOCATION
#
# Estrategia en 8 etapas (cascada):
#   1. Internacionales (keyword exacta)
#   2. Estado explícito en el texto
#   3. Abreviaturas de estado
#   4. Autopistas / rutas conocidas
#   5. Ciudades / municipios
#   6. TADs / TARs / ASAs (terminales PEMEX)
#   7. Instalaciones PEMEX (refinerías, plataformas, complejos)
#   8. Fuzzy matching de último recurso
# ════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────
# UTILIDADES
# ─────────────────────────────────────────────────────────────

def _norm(texto: str) -> str:
    """Normaliza para comparación: mayúsculas, sin acentos, sin puntuación extra."""
    if not texto or str(texto).strip() in ("", "-", "nan", "N/A", "NO ESPECIFICADO"):
        return ""
    t = unidecode(str(texto)).upper().strip()
    t = re.sub(r"[;:'\"\[\]\(\)\{\}*#!]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 1 – ESTADOS DE MÉXICO
# ─────────────────────────────────────────────────────────────

ESTADOS_MX = OrderedDict([
    ("BAJA CALIFORNIA SUR",   "BAJA CALIFORNIA SUR"),
    ("QUINTANA ROO",          "QUINTANA ROO"),
    ("SAN LUIS POTOSI",       "SAN LUIS POTOSI"),
    ("CIUDAD DE MEXICO",      "CIUDAD DE MEXICO"),
    ("ESTADO DE MEXICO",      "ESTADO DE MEXICO"),
    ("BAJA CALIFORNIA",       "BAJA CALIFORNIA"),
    ("NUEVO LEON",            "NUEVO LEON"),
    ("AGUASCALIENTES",        "AGUASCALIENTES"),
    ("CAMPECHE",              "CAMPECHE"),
    ("CHIAPAS",               "CHIAPAS"),
    ("CHIHUAHUA",             "CHIHUAHUA"),
    ("COAHUILA",              "COAHUILA"),
    ("COLIMA",                "COLIMA"),
    ("DURANGO",               "DURANGO"),
    ("GUANAJUATO",            "GUANAJUATO"),
    ("GUERRERO",              "GUERRERO"),
    ("HIDALGO",               "HIDALGO"),
    ("JALISCO",               "JALISCO"),
    ("MICHOACAN",             "MICHOACAN"),
    ("MORELOS",               "MORELOS"),
    ("NAYARIT",               "NAYARIT"),
    ("OAXACA",                "OAXACA"),
    ("PUEBLA",                "PUEBLA"),
    ("QUERETARO",             "QUERETARO"),
    ("SINALOA",               "SINALOA"),
    ("SONORA",                "SONORA"),
    ("TABASCO",               "TABASCO"),
    ("TAMAULIPAS",            "TAMAULIPAS"),
    ("TLAXCALA",              "TLAXCALA"),
    ("VERACRUZ",              "VERACRUZ"),
    ("YUCATAN",               "YUCATAN"),
    ("ZACATECAS",             "ZACATECAS"),
])


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 2 – ABREVIATURAS DE ESTADO
# ─────────────────────────────────────────────────────────────

ABREVIATURAS = {
    "AGS":            "AGUASCALIENTES",
    "BC":             "BAJA CALIFORNIA",
    "B.C":            "BAJA CALIFORNIA",
    "B.C.": "BAJA CALIFORNIA",
    "BCN":            "BAJA CALIFORNIA",
    "BCS":            "BAJA CALIFORNIA SUR",
    "B.C.S":          "BAJA CALIFORNIA SUR",
    "CAMP":           "CAMPECHE",
    "CHIS":           "CHIAPAS",
    "CHIH":           "CHIHUAHUA",
    "CH":             "CHIHUAHUA",
    "COAH":           "COAHUILA",
    "COL":            "COLIMA",
    "CDMX":           "CIUDAD DE MEXICO",
    "D.F.": "CIUDAD DE MEXICO",
    "D.F":            "CIUDAD DE MEXICO",
    "DF":             "CIUDAD DE MEXICO",
    "DGO":            "DURANGO",
    "GTO":            "GUANAJUATO",
    "GRO":            "GUERRERO",
    "HGO":            "HIDALGO",
    "JAL":            "JALISCO",
    "MEX":            "ESTADO DE MEXICO",
    "EDOMEX":         "ESTADO DE MEXICO",
    "EDO MEX":        "ESTADO DE MEXICO",
    "EDO. MEX":       "ESTADO DE MEXICO",
    "EDO DE MEX":     "ESTADO DE MEXICO",
    "EDO. DE MEX":    "ESTADO DE MEXICO",
    "EDO DE MEXICO":  "ESTADO DE MEXICO",
    "EDO. DE MEXICO": "ESTADO DE MEXICO",
    "EDO. MEXICO":    "ESTADO DE MEXICO",
    "MICH":           "MICHOACAN",
    "MOR":            "MORELOS",
    "NAY":            "NAYARIT",
    "NL":             "NUEVO LEON",
    "NL.": "NUEVO LEON",
    "N.L":            "NUEVO LEON",
    "N.L.": "NUEVO LEON",
    "OAX":            "OAXACA",
    "PUE":            "PUEBLA",
    "QRO":            "QUERETARO",
    "Q. ROO":         "QUINTANA ROO",
    "QROO":           "QUINTANA ROO",
    "Q.ROO":          "QUINTANA ROO",
    "Q ROO":          "QUINTANA ROO",
    "SLP":            "SAN LUIS POTOSI",
    "S.L.P":          "SAN LUIS POTOSI",
    "SIN":            "SINALOA",
    "SON":            "SONORA",
    "TAB":            "TABASCO",
    "TAM":            "TAMAULIPAS",
    "TAMPS":          "TAMAULIPAS",
    "TLAX":           "TLAXCALA",
    "VER":            "VERACRUZ",
    "VER.": "VERACRUZ",
    "VERACRZ":        "VERACRUZ",
    "YUC":            "YUCATAN",
    "ZAC":            "ZACATECAS",
    "GTEZ":           "CHIAPAS",    # Tuxtla Gtez.
}


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 3 – INTERNACIONALES
# ─────────────────────────────────────────────────────────────

INTERNACIONAL_KEYWORDS = {
    # USA
    "USA":                    ("NO ESPECIFICADO", "USA"),
    "EUA":                    ("NO ESPECIFICADO", "USA"),
    "EEUU":                   ("NO ESPECIFICADO", "USA"),
    "UNITED STATES":          ("NO ESPECIFICADO", "USA"),
    "TEXAS":                  ("TEXAS",            "USA"),
    "TX":                     ("TEXAS",            "USA"),
    "HOUSTON":                ("TEXAS",            "USA"),
    "GALVESTON":              ("TEXAS",            "USA"),
    "BOLIVAR ROADS":          ("TEXAS",            "USA"),
    "BEAUMONT":               ("TEXAS",            "USA"),
    "PORT ARTHUR":            ("TEXAS",            "USA"),
    "PORT ARTUR":             ("TEXAS",            "USA"),
    "PASADENA":               ("TEXAS",            "USA"),
    "PAULSBORO":              ("NEW JERSEY",       "USA"),
    "NUEVA JERSEY":           ("NEW JERSEY",       "USA"),
    "NEW JERSEY":             ("NEW JERSEY",       "USA"),
    "LOUISIANA":              ("LOUISIANA",        "USA"),
    "LAKE CHARLES":           ("LOUISIANA",        "USA"),
    "TERMINAL DE ST. CHARLES VALERO": ("LOUISIANA", "USA"),
    "SW PASS":                ("LOUISIANA",        "USA"),
    "SOUTHWEST PASS":         ("LOUISIANA",        "USA"),
    "CALIFORNIA":             ("CALIFORNIA",       "USA"),
    "SAN DIEGO":              ("CALIFORNIA",       "USA"),
    "SAN FRANCISCO":          ("CALIFORNIA",       "USA"),
    "BROWNSVILLE":            ("TEXAS",            "USA"),
    # América Latina
    "PANAMA":                 ("PANAMA",           "PANAMA"),
    "BALBOA":                 ("PANAMA",           "PANAMA"),
    "GATUN":                  ("PANAMA",           "PANAMA"),
    "COLON PANAMA":           ("COLON",            "PANAMA"),
    "CHILE":                  ("NO ESPECIFICADO",  "CHILE"),
    "TALCAHUANO":             ("BIOBIO",           "CHILE"),
    "BAHIA CONCEPCION CHILE": ("BIOBIO",           "CHILE"),
    "ARUBA":                  ("ARUBA",            "ARUBA"),
    "CANADA":                 ("NO ESPECIFICADO",  "CANADA"),
    "HALIFAX":                ("NUEVA ESCOCIA",    "CANADA"),
    # Europa / Asia
    "ALEMANIA":               ("NO ESPECIFICADO",  "ALEMANIA"),
    "AMSTERDAM":              ("HOLANDA SEPTENTRIONAL", "PAISES BAJOS"),
    "TERMINAL AMSTERDAM":     ("HOLANDA SEPTENTRIONAL", "PAISES BAJOS"),
    "VOPAK WESTPOORT":        ("HOLANDA SEPTENTRIONAL", "PAISES BAJOS"),
    "SINGAPORE":              ("SINGAPUR",         "SINGAPUR"),
    "SINGAPUR":               ("SINGAPUR",         "SINGAPUR"),
    "TANKSTORE TERMINAL":     ("SINGAPUR",         "SINGAPUR"),
    "NIPAH":                  ("NO ESPECIFICADO",  "INDONESIA"),
    "INDONESIA":              ("NO ESPECIFICADO",  "INDONESIA"),
    "ATHENS":                 ("ÁTICA",            "GRECIA"),
    "OVERSEAS ATHENS":        ("ÁTICA",            "GRECIA"),
    # Mar / Offshore
    "REGION MARINA":          ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "GOLFO DE MEXICO":        ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "AGUAS DEL GOLFO":        ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "BAHIA DE CAMPECHE":      ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "PACIFICO NORTE":         ("PACIFICO NORTE",   "MAR INTERNACIONAL"),
    "PACIFIC AREA LIGHTERING":("PACIFICO NORTE",   "MAR INTERNACIONAL"),
    "NOHOCH":                 ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "CANTARELL":              ("GOLFO DE MEXICO",  "MAR INTERNACIONAL"),
    "MARITIMO":               ("NO ESPECIFICADO",  "MAR INTERNACIONAL"),
    "BERING GAS":             ("NO ESPECIFICADO",  "MAR INTERNACIONAL"),
    "LPG BERING":             ("NO ESPECIFICADO",  "MAR INTERNACIONAL"),
}

CAPITALES_POR_PAIS = {
    "PANAMA":        "CIUDAD DE PANAMA",
    "CHILE":         "SANTIAGO",
    "ALEMANIA":      "BERLIN",
    "SINGAPUR":      "SINGAPUR",
    "PAISES BAJOS":  "AMSTERDAM",
    "CANADA":        "NO ESPECIFICADO",
    "INDONESIA":     "NO ESPECIFICADO",
    "GRECIA":        "ATENAS",
    "ARUBA":         "ORANJESTAD",
}


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 4 – AUTOPISTAS / RUTAS
# ─────────────────────────────────────────────────────────────

AUTOPISTAS = {
    "MEX-QRO":                       "QUERETARO",
    "MEXICO-QUERETARO":              "QUERETARO",
    "PALMILLAS-APASEO":              "QUERETARO",
    "SAN LUIS POTOSI-QUERETARO":     "SAN LUIS POTOSI",
    "ARCO NORTE":                    "HIDALGO",
    "MEX-TUXPAN":                    "VERACRUZ",
    "CORDOBA-VERACRUZ":              "VERACRUZ",
    "CORDOBA-PUEBLA":                "VERACRUZ",
    "COATZACOALCOS-VILLAHERMOSA":    "VERACRUZ",
    "TINAJA-ACAYUCAN":               "VERACRUZ",
    "LA TINAJA-ACAYUCAN":            "VERACRUZ",
    "PAJARITOS-VILLAHERMOSA":        "VERACRUZ",
    "CUERNAVACA-IGUALA":             "MORELOS",
    "CUERNAVACA-ACAPULCO":           "GUERRERO",
    "ACAPULCO-ZIHUATANEJO":          "GUERRERO",
    "LEON-SALAMANCA":                "GUANAJUATO",
    "LEON-AGUASCALIENTES":           "JALISCO",
    "IRAPUATO-SALAMANCA":            "GUANAJUATO",
    "SALTILLO-MONTERREY":            "COAHUILA",
    "MONTERREY-CD VICTORIA":         "NUEVO LEON",
    "MONTERREY-REYNOSA":             "NUEVO LEON",
    "CADEREYTA-MONTERREY":           "NUEVO LEON",
    "TORREON-DURANGO":               "DURANGO",
    "MAZATLAN-DURANGO":              "SINALOA",
    "MOCHIS-NAVOJOA":                "SINALOA",
    "TIJUANA-MEXICALI":              "BAJA CALIFORNIA",
    "TIJUANA-TECATE":                "BAJA CALIFORNIA",
    "LAZARO CARDENAS-URUAPAN":       "MICHOACAN",
    "PATZCUARO-URUAPAN":             "MICHOACAN",
    "CIUDAD VALLES-TAMPICO":         "SAN LUIS POTOSI",
    "CIUDAD VICTORIA-MONTERREY":     "TAMAULIPAS",
    "ARRIAGA-OCOZOCOAUTLA":          "CHIAPAS",
    "ARRIAGA-OCOZOCUAUTLA":          "CHIAPAS",
    "TAPACHULA-PUERTO SAN BENITO":   "CHIAPAS",
    "SALINA CRUZ-MATIAS ROMERO":     "OAXACA",
    "MITLA-TEHUANTEPEC":             "OAXACA",
    "TRANSISTMICA":                  "OAXACA",
    "TRANSISMICA":                   "OAXACA",
    "TEPIC-PUENTE TALISMAN":         "NAYARIT",
    "IXTEPEC-TEHUANTEPEC":           "OAXACA",
    "TRANSPENINSULAR":               "BAJA CALIFORNIA SUR",
    "TRANSPENINSULAR BENITO JUAREZ": "BAJA CALIFORNIA SUR",
    "CARRETERA FEDERAL 40":          "SINALOA",
    "CARRETERA MEXICO 40":           "SINALOA",
    "CARR. LA RUMOROSA":           "BAJA CALIFORNIA"
}


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 5 – CIUDADES / MUNICIPIOS
# ─────────────────────────────────────────────────────────────

CIUDADES = {
    #Colima
    "MANZANILLO":              "COLIMA",
    # Tamaulipas
    "CIUDAD MADERO":           "TAMAULIPAS",
    "CD MADERO":               "TAMAULIPAS",
    "CD. MADERO":              "TAMAULIPAS",
    "MADERO":                  "TAMAULIPAS",
    "ALTAMIRA":                "TAMAULIPAS",
    "TAMPICO":                 "TAMAULIPAS",
    "REYNOSA":                 "TAMAULIPAS",
    "MATAMOROS":               "TAMAULIPAS",
    "CIUDAD VICTORIA":         "TAMAULIPAS",
    "CD VICTORIA":             "TAMAULIPAS",
    "CD. VICTORIA":            "TAMAULIPAS",
    "NUEVO LAREDO":            "TAMAULIPAS",
    "MANTE":                   "TAMAULIPAS",
    "CIUDAD MANTE":            "TAMAULIPAS",
    # Veracruz
    "COATZACOALCOS":           "VERACRUZ",
    "COATZACOALCO":            "VERACRUZ",
    "COATZACOLACOS":           "VERACRUZ",
    "COSOLOACAQUE":            "VERACRUZ",
    "COSOLEACAQUE":            "VERACRUZ",
    "MINATITLAN":              "VERACRUZ",
    "POZA RICA":               "VERACRUZ",
    "TUXPAN":                  "VERACRUZ",
    "CORDOBA":                 "VERACRUZ",
    "ORIZABA":                 "VERACRUZ",
    "XALAPA":                  "VERACRUZ",
    "COSAMALOAPAN":            "VERACRUZ",
    "ACAYUCAN":                "VERACRUZ",
    "NANCHITAL":               "VERACRUZ",
    "LAS CHOAPAS":             "VERACRUZ",
    "MARTINEZ DE LA TORRE":    "VERACRUZ",
    "PEROTE":                  "VERACRUZ",
    "TIERRA BLANCA":           "VERACRUZ",
    "COTAXTLA":                "VERACRUZ",
    "PAJARITOS":               "VERACRUZ",
    "PUERTO DE VERACRUZ":      "VERACRUZ",
    "BAJOS DE LA GALLEGA":     "VERACRUZ",
    "CE BAJOS DE LA GALLEGA":  "VERACRUZ",
    "ESCAMELA":                "VERACRUZ",
    "SAYULA DE ALEMAN":        "VERACRUZ",
    "CANGREJERA":              "VERACRUZ",
    "MISANTLA":                "VERACRUZ",
    "COTAXTLA":                "VERACRUZ",
    # Nuevo León
    "MONTERREY":               "NUEVO LEON",
    "APODACA":                 "NUEVO LEON",
    "CADEREYTA":               "NUEVO LEON",
    "CADEREYTA JIMENEZ":       "NUEVO LEON",
    "GARCIA":                  "NUEVO LEON",
    "GUADALUPE":               "NUEVO LEON",
    "SANTA CATARINA":          "NUEVO LEON",
    "LINARES":                 "NUEVO LEON",
    "CIENEGA DE FLORES":       "NUEVO LEON",
    # Ciudad de México
    "AZCAPOTZALCO":            "CIUDAD DE MEXICO",
    "IZTAPALAPA":              "CIUDAD DE MEXICO",
    "GUSTAVO A. MADERO":       "CIUDAD DE MEXICO",
    "TLALPAN":                 "CIUDAD DE MEXICO",
    "XOCHIMILCO":              "CIUDAD DE MEXICO",
    "CUAJIMALPA":              "CIUDAD DE MEXICO",
    "ALVARO OBREGON":          "CIUDAD DE MEXICO",
    "MIGUEL HIDALGO":          "CIUDAD DE MEXICO",
    "ANLL":                    "CIUDAD DE MEXICO",
    "18 DE MARZO":             "CIUDAD DE MEXICO",
    # Estado de México
    "TOLUCA":                  "ESTADO DE MEXICO",
    "ECATEPEC":                "ESTADO DE MEXICO",
    "NAUCALPAN":               "ESTADO DE MEXICO",
    "TLALNEPANTLA":            "ESTADO DE MEXICO",
    "ATIZAPAN":                "ESTADO DE MEXICO",
    "CUAUTITLAN IZCALLI":      "ESTADO DE MEXICO",
    "CUAUTITLAN":              "ESTADO DE MEXICO",
    "TEPOTZOTLAN":             "ESTADO DE MEXICO",
    "TULTITLAN":               "ESTADO DE MEXICO",
    "ATLACOMULCO":             "ESTADO DE MEXICO",
    "ACAMBAY":                 "ESTADO DE MEXICO",
    "JILOTEPEC":               "ESTADO DE MEXICO",
    # Campeche
    "CIUDAD DEL CARMEN":       "CAMPECHE",
    "CD. DEL CARMEN":          "CAMPECHE",
    "CD DEL CARMEN":           "CAMPECHE",
    "CARMEN":                  "CAMPECHE",
    "CALKINI":                 "CAMPECHE",
    "PUERTO DE LERMA":         "CAMPECHE",
    "LERMA":                   "CAMPECHE",
    # Tabasco
    "VILLAHERMOSA":            "TABASCO",
    "CARDENAS":                "TABASCO",
    "CUNDUACAN":               "TABASCO",
    "PARAISO":                 "TABASCO",
    "DOS BOCAS":               "TABASCO",
    "CENTRO":                  "TABASCO",
    # Jalisco
    "GUADALAJARA":             "JALISCO",
    "ZAPOPAN":                 "JALISCO",
    "TLAQUEPAQUE":             "JALISCO",
    "PUERTO VALLARTA":         "JALISCO",
    "CD GUZMAN":               "JALISCO",
    "LAGOS DE MORENO":         "JALISCO",
    "ENCARNACION DIAZ":        "JALISCO",
    "EL CASTILLO":             "JALISCO",
    # Guanajuato
    "LEON":                    "GUANAJUATO",
    "CELAYA":                  "GUANAJUATO",
    "IRAPUATO":                "GUANAJUATO",
    "SALAMANCA":               "GUANAJUATO",
    "SILAO":                   "GUANAJUATO",
    "APASEO EL GRANDE":        "GUANAJUATO",
    "SAN JOSE DE ITURBIDE":    "GUANAJUATO",
    "SAN JOSE ITURBIDE":       "GUANAJUATO",
    "SAN MIGUEL DE ALLENDE":   "GUANAJUATO",
    # Guerrero
    "BAHIA DE ACAPULCO":       "GUERRERO",
    "ACAPULCO":                "GUERRERO",
    "ZIHUATANEJO":             "GUERRERO",
    "CHILPANCINGO":            "GUERRERO",
    "IGUALA":                  "GUERRERO",
    "TAXCO":                   "GUERRERO",
    "SAN MARCOS":              "GUERRERO",
    # Puebla
    "SAN MARTIN TEXMELUCAN":   "PUEBLA",
    "TEZIUTLAN":               "PUEBLA",
    "TEHUACAN":                "PUEBLA",
    "ATLIXCO":                 "PUEBLA",
    "ACTAZINGO":               "PUEBLA",
    "MIAHUATLAN":              "PUEBLA",
    # Querétaro
    "SANTIAGO DE QUERETARO":   "QUERETARO",
    "SAN JUAN DEL RIO":        "QUERETARO",
    "PALMILLAS":               "QUERETARO",
    # Hidalgo
    "PACHUCA":                 "HIDALGO",
    "TULA DE ALLENDE":         "HIDALGO",
    "TULA":                    "HIDALGO",
    "ACTOPAN":                 "HIDALGO",
    "SINGUILUCAN":             "HIDALGO",
    "TULANCINGO":              "HIDALGO",
    # Coahuila
    "SALTILLO":                "COAHUILA",
    "TORREON":                 "COAHUILA",
    "MONCLOVA":                "COAHUILA",
    "PIEDRAS NEGRAS":          "COAHUILA",
    "RAMOS ARIZPE":            "COAHUILA",
    "SABINAS":                 "COAHUILA",
    # San Luis Potosí
    "MATEHUALA":               "SAN LUIS POTOSI",
    "CIUDAD VALLES":           "SAN LUIS POTOSI",
    "CD VALLES":               "SAN LUIS POTOSI",
    "CD DEL MAIZ":             "SAN LUIS POTOSI",
    # Oaxaca
    "SALINA CRUZ":             "OAXACA",
    "JUCHITAN":                "OAXACA",
    "IXTEPEC":                 "OAXACA",
    "TEHUANTEPEC":             "OAXACA",
    "MATIAS ROMERO":           "OAXACA",
    "PINOTEPA NACIONAL":       "OAXACA",
    # Sinaloa
    "MAZATLAN":                "SINALOA",
    "CULIACAN":                "SINALOA",
    "LOS MOCHIS":              "SINALOA",
    "AHOME":                   "SINALOA",
    "NAVOLATO":                "SINALOA",
    "TOPOLOBAMPO":             "SINALOA",
    # Sonora
    "HERMOSILLO":              "SONORA",
    "CAJEME":                  "SONORA",
    "CIUDAD OBREGON":          "SONORA",
    "CD OBREGON":              "SONORA",
    "GUAYMAS":                 "SONORA",
    "NOGALES":                 "SONORA",
    "EMPALME":                 "SONORA",
    "CANANEA":                 "SONORA",
    "NAVOJOA":                 "SONORA",
    "IMURIS":                  "SONORA",
    "AGUA PRIETA":             "SONORA",
    "MAGDALENA":               "SONORA",
    # Chihuahua
    "CIUDAD JUAREZ":           "CHIHUAHUA",
    "CD JUAREZ":               "CHIHUAHUA",
    "DELICIAS":                "CHIHUAHUA",
    "CUAUHTEMOC":              "CHIHUAHUA",
    "HIDALGO DEL PARRAL":      "CHIHUAHUA",
    "CAMARGO":                 "CHIHUAHUA",
    # Chiapas
    "TUXTLA GUTIERREZ":        "CHIAPAS",
    "TAPACHULA":               "CHIAPAS",
    "ARRIAGA":                 "CHIAPAS",
    "OCOZOCOAUTLA":            "CHIAPAS",
    "OCOZOCUAUTLA":            "CHIAPAS",
    "ACACOYAGUA":              "CHIAPAS",
    "HUIXTLA":                 "CHIAPAS",
    # Morelos
    "CUERNAVACA":              "MORELOS",
    "CUAUTLA":                 "MORELOS",
    # Yucatán
    "MERIDA":                  "YUCATAN",
    "PROGRESO":                "YUCATAN",
    "KANTUNIL":                "YUCATAN",
    # Quintana Roo
    "CANCUN":                  "QUINTANA ROO",
    "PLAYA DEL CARMEN":        "QUINTANA ROO",
    "CHETUMAL":                "QUINTANA ROO",
    "COZUMEL":                 "QUINTANA ROO",
    # Baja California
    "TIJUANA":                 "BAJA CALIFORNIA",
    "MEXICALI":                "BAJA CALIFORNIA",
    "ENSENADA":                "BAJA CALIFORNIA",
    "TECATE":                  "BAJA CALIFORNIA",
    "ROSARITO":                "BAJA CALIFORNIA",
    # Baja California Sur
    "SAN JOSE DEL CABO":       "BAJA CALIFORNIA SUR",
    "CABO SAN LUCAS":          "BAJA CALIFORNIA SUR",
    "LA PAZ":                  "BAJA CALIFORNIA SUR",
    "LOS BARRILES":            "BAJA CALIFORNIA SUR",
    # Durango
    "LERDO":                   "DURANGO",
    "GOMEZ PALACIO":           "DURANGO",
    "AGUILERA":                "DURANGO",
    "VICENTE GUERRERO":        "DURANGO",
    # Michoacán
    "LAZARO CARDENAS":         "MICHOACAN",
    "URUAPAN":                 "MICHOACAN",
    "PATZCUARO":               "MICHOACAN",
    "MORELIA":                 "MICHOACAN",
    "ZAMORA":                  "MICHOACAN",
    # Nayarit / Zacatecas / Otros
    "TEPIC":                   "NAYARIT",
    "CALPULALPAN":             "TLAXCALA",
    "CONCEPCION DEL ORO":      "ZACATECAS",
    "FRESNILLO":               "ZACATECAS",
    "COSIO":                   "AGUASCALIENTES",
    "CHAPALILLA":              "NAYARIT",
    "COMPOSTELA":              "NAYARIT",
    # Aeropuertos / Terminales especiales
    "AEROPUERTO DE SAN JOSE DEL CABO": "BAJA CALIFORNIA SUR",
}


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 6 – TADs / TARs / ASAs  →  estado
# ─────────────────────────────────────────────────────────────

TADS = {
    # (se buscan como substring; la key más larga gana)
    "TAD ACAPULCO":             "GUERRERO",
    "TAD AGUASCALIENTES":       "AGUASCALIENTES",
    "TAD CAMPECHE":             "CAMPECHE",
    "TAD CANCUN":               "QUINTANA ROO",
    "TAD CADEREYTA":            "NUEVO LEON",
    "TAD CD OBREGON":           "SONORA",
    "TAD CD VICTORIA":          "TAMAULIPAS",
    "TAD CIUDAD VICTORIA":      "TAMAULIPAS",
    "TAD CD JUAREZ":            "CHIHUAHUA",
    "TAD STA. CATARINA":        "NUEVO LEON",
    "TAD CIUDAD JUAREZ":        "CHIHUAHUA",
    "TAD CIUDAD MADERO":        "TAMAULIPAS",
    "TAD CHIHUAHUA":            "CHIHUAHUA",
    "TAD COATZACOALCOS":        "VERACRUZ",
    "TAD CUERNAVACA":           "MORELOS",
    "TAD CUAUTLA":              "MORELOS",
    "TAD DURANGO":              "DURANGO",
    "TAD EL CASTILLO":          "JALISCO",
    "TAD CASTILLO":             "JALISCO",
    "TAD ESCAMELA":             "VERACRUZ",
    "TAD GOMEZ PALACIO":        "DURANGO",
    "TAD GUADALAJARA":          "JALISCO",
    "TAD GUAYMAS":              "SONORA",
    "TAD HERMOSILLO":           "SONORA",
    "TAD HIDALGO DEL PARRAL":   "CHIHUAHUA",
    "TAD IGUALA":               "GUERRERO",
    "TAD IRAPUATO":             "GUANAJUATO",
    "TAD LAZARO CARDENAS":      "MICHOACAN",
    "TAD LEON":                 "GUANAJUATO",
    "TAD MADERO":               "TAMAULIPAS",
    "TAD MAGDALENA":            "SONORA",
    "TAD MANZANILLO":           "COLIMA",
    "TAD MATEHUALA":            "SAN LUIS POTOSI",
    "TAD MAZATLAN":             "SINALOA",
    "TAD MEXICALI":             "BAJA CALIFORNIA",
    "TAD MIAHUATLAN":           "PUEBLA",
    "TAD MINATITLAN":           "VERACRUZ",
    "TAD MONCLOVA":             "COAHUILA",
    "TAD MORELIA":              "MICHOACAN",
    "TAD MONTERREY":            "NUEVO LEON",
    "TAD NOGALES":              "SONORA",
    "TAD NUEVO LAREDO":         "TAMAULIPAS",
    "TAD OAXACA":               "OAXACA",
    "TAD PACHUCA":              "HIDALGO",
    "TAD PAJARITOS":            "VERACRUZ",
    "TAD PEROTE":               "VERACRUZ",
    "TAD POZA RICA":            "VERACRUZ",
    "TAD PROGRESO":             "YUCATAN",
    "TAD PUEBLA":               "PUEBLA",
    "TAD QUERETARO":            "QUERETARO",
    "TAD REYNOSA":              "TAMAULIPAS",
    "TAD SABINAS":              "COAHUILA",
    "TAD SALINA CRUZ":          "OAXACA",
    "TAD SALAMANCA":            "GUANAJUATO",
    "TAD SALTILLO":             "COAHUILA",
    "TAD SAN JOSE ITURBIDE":    "GUANAJUATO",
    "TAD SAN LUIS POTOSI":      "SAN LUIS POTOSI",
    "TAD SANTA CATARINA":       "NUEVO LEON",
    "TAD STA. CATARINA":         "NUEVO LEON",
    "TAD TAMPICO":              "TAMAULIPAS",
    "TAD TAPACHULA":            "CHIAPAS",
    "TAD TEPEIXTLES":           "COLIMA",
    "TAD TIERRA BLANCA":        "VERACRUZ",
    "TAD TOLUCA":               "ESTADO DE MEXICO",
    "TAD TOPOLOBAMPO":          "SINALOA",
    "TAD TULA":                 "HIDALGO",
    "TAD TUXTLA GUTIERREZ":     "CHIAPAS",
    "TAD TUXPAN":               "VERACRUZ",
    "TAD ITZOIL TUXPAN":        "VERACRUZ",
    "TAD URUAPAN":              "MICHOACAN",
    "TAD VERACRUZ":             "VERACRUZ",
    "TAD VILLAHERMOSA":         "TABASCO",
    "TAD ZACATECAS":            "ZACATECAS",
    "TAD ZAMORA":               "MICHOACAN",
    "TAD ZAPOPAN":              "JALISCO",
    "TAD LA PAZ":               "BAJA CALIFORNIA SUR",
    "TAD 18 DE MARZO":          "CIUDAD DE MEXICO",
    "TAD 18 MARZO":             "CIUDAD DE MEXICO",
    "TAD AZCAPOTZALCO":         "CIUDAD DE MEXICO",
    # TAR (misma lógica)
    "TAR ACAPULCO":             "GUERRERO",
    "TAR DE CD. JUAREZ":           "CHIHUAHUA",
    "TAR CADEREYTA":            "NUEVO LEON",
    "TAR CD MADERO":            "TAMAULIPAS",
    "TAR CIUDAD MADERO":        "TAMAULIPAS",
    "TAR IRAPUATO":             "GUANAJUATO",
    "TAR LAZARO CARDENAS":      "MICHOACAN",
    "TAR MANZANILLO":           "COLIMA",
    "TAR DE JALAPA.":            "VERACRUZ",
    "TAR MAZATLAN":             "SINALOA",
    "TAR MINATITLAN":           "VERACRUZ",
    "TAR MORELIA":              "MICHOACAN",
    "TAR OAXACA":               "OAXACA",
    "TAR PACHUCA":              "HIDALGO",
    "TAR PAJARITOS":            "VERACRUZ",
    "TAR PUEBLA":               "PUEBLA",
    "TAR QUERETARO":            "QUERETARO",
    "TAR SALINA CRUZ":          "OAXACA",
    "TAR SALTILLO":             "COAHUILA",
    "TAR SANTA CATARINA":       "NUEVO LEON",
    "TAR TAPACHULA":            "CHIAPAS",
    "TAR TOPOLOBAMPO":          "SINALOA",
    "TAR TULA":                 "HIDALGO",
    "TAR TUXTLA GUTIERREZ":     "CHIAPAS",
    "TAR URUAPAN":              "MICHOACAN",
    "TAR VERACRUZ":             "VERACRUZ",
    "TAR VILLAHERMOSA":         "TABASCO",
    "TAR ZAMORA":               "MICHOACAN",
    # TASP (Terminal Almacenamiento y Suministro de Pemex)
    "TASP PAJARITOS":           "VERACRUZ",
    "TASP MADERO":              "TAMAULIPAS",
    # Terminales marítimas
    "TERMINAL MARITIMA MADERO": "TAMAULIPAS",
    "TERMINAL MADERO":          "TAMAULIPAS",
    "TERMINAL PAJARITOS":       "VERACRUZ",
    "ESTACION DE BOMBEO LINARES": "NUEVO LEON",
}


# ─────────────────────────────────────────────────────────────
# CATÁLOGO 7 – INSTALACIONES PEMEX
# ─────────────────────────────────────────────────────────────

INSTALACIONES_PEMEX = {
    "REFINERIA CADEREYTA":                          "NUEVO LEON",
    "REFINERIA DE CADEREYTA":                       "NUEVO LEON",
    "REFINERIA TULA":                               "HIDALGO",
    "REFINERIA SALAMANCA":                          "GUANAJUATO",
    "REFINERIA SALINA CRUZ":                        "OAXACA",
    "REFINERIA MINATITLAN":                         "VERACRUZ",
    "REFINERIA CIUDAD MADERO":                      "TAMAULIPAS",
    "REFINERIA CD MADERO":                          "TAMAULIPAS",
    "COMPLEJO PETROQUIMICO SAN MARTIN TEXMELUCAN":  "PUEBLA",
    "COMPLEJO PETROQUIMICOS MARTIN TEX.MELUCAN":    "PUEBLA",
    "COMPLEJO PETROQUIMICOS MARTIN TEX.":           "PUEBLA",
    "SAN MARTIN TEXMELUCAN":                        "PUEBLA",
    "CPQ SAN MARTIN TEXMELUCAN":                    "PUEBLA",
    "COMPLEJO SAN MARTIN TEXM":                     "PUEBLA",
    "COMPLEJO PETROQUIMICO CANGREJERA":             "VERACRUZ",
    "COMPLEJO PETROQUIMICO PAJARITOS":              "VERACRUZ",
    "COMPLEJO PROCESADOR DE GAS REYNOSA":           "TAMAULIPAS",
    "PLATAFORMA AKAL":                              "CAMPECHE",
    "AKAL-C":                                       "CAMPECHE",
    "AKAL C":                                       "CAMPECHE",
    "NOHOCH-A":                                     "CAMPECHE",
    "CHALAN PEMEX 538":                             "CAMPECHE",
    "ACTIVO DE PRODUCCION CANTARELL":               "CAMPECHE",
    "DOS BOCAS":                                    "TABASCO",
    "PUERTO DOS BOCAS":                             "TABASCO",
}


# ─────────────────────────────────────────────────────────────
# FUNCIÓN PRINCIPAL clasificar_estado()
# ─────────────────────────────────────────────────────────────

_SORTED_INTL   = sorted(INTERNACIONAL_KEYWORDS.items(), key=lambda x: len(x[0]), reverse=True)
_SORTED_TADS   = sorted(TADS.items(),  key=lambda x: len(x[0]), reverse=True)
_SORTED_INSTAL = sorted(INSTALACIONES_PEMEX.items(), key=lambda x: len(x[0]), reverse=True)
_SORTED_CIUD   = sorted(CIUDADES.items(), key=lambda x: len(x[0]), reverse=True)
_SORTED_ABREV  = sorted(ABREVIATURAS.items(), key=lambda x: len(x[0]), reverse=True)
_SORTED_RUTA   = sorted(AUTOPISTAS.items(), key=lambda x: len(x[0]), reverse=True)

def clasificar_estado(location_raw) -> dict:
    """
    Clasifica una location libre. Devuelve dict con:
      estado    : nombre del estado (MAYÚSCULAS) o 'NO ESPECIFICADO'
      pais      : 'MEXICO', 'USA', 'PANAMA', etc.  o 'NO ESPECIFICADO'
      metodo    : etapa que produjo el resultado
      confianza : 'ALTA' | 'MEDIA' | 'BAJA' | 'N/A'

    Cascada de 8 etapas:
      1. Internacional (keywords exactas)
      2. Estado explícito en texto
      3. Abreviaturas de estado
      4. Autopistas / rutas
      5. Ciudades / municipios
      6. TADs / TARs / terminales PEMEX
      7. Instalaciones PEMEX (refinerías, complejos, plataformas)
      8. Fuzzy matching (último recurso)
    """
    texto = _norm(location_raw)

    if not texto:
        return {"estado": "NO ESPECIFICADO", "pais": "NO ESPECIFICADO",
                "metodo": "vacio", "confianza": "N/A"}

    # ── ETAPA 1: Internacionales ──────────────────────────────────────────
    for keyword, (est_int, pais_int) in _SORTED_INTL:
        kw = _norm(keyword)
        # Guardrail: evitar falsos positivos para tokens cortos ambiguos
        if kw in ("LA", "TX", "CALIFORNIA") and not re.search(
                r'\b(USA|EUA|EEUU|UNITED STATES)\b', texto):
            continue
        if kw == "BAHIA DE CAMPECHE" and "CAMPECHE" in texto and not re.search(
                r'\b(REGION MARINA|GOLFO|PLATAFORMA|ACTIVO)\b', texto):
            continue
        if re.search(r'\b' + re.escape(kw) + r'\b', texto):
            # Si hay señales de que es México → saltar
            texto_sin_golfo = re.sub(r'GOLFO\s+DE\s+MEXICO', '', texto)
            if re.search(r'\b(MEXICO|MEX)\b', texto_sin_golfo) or \
               any(e in texto_sin_golfo for e in ESTADOS_MX):
                continue
            estado_final = est_int if est_int != "NO ESPECIFICADO" \
                           else CAPITALES_POR_PAIS.get(pais_int, "NO ESPECIFICADO")
            return {"estado": estado_final, "pais": pais_int,
                    "metodo": "internacional", "confianza": "ALTA"}

    # ── ETAPA 2: Estado explícito ─────────────────────────────────────────
    for est_clave, est_oficial in ESTADOS_MX.items():
        if est_clave in texto:
            return {"estado": est_oficial, "pais": "MEXICO",
                    "metodo": "estado_explicito", "confianza": "ALTA"}

    # ── ETAPA 3: Abreviaturas ─────────────────────────────────────────────
    for abrev, estado_abrev in _SORTED_ABREV:
        abrev_n = _norm(abrev)
        if re.search(r'\b' + re.escape(abrev_n) + r'\b', texto):
            return {"estado": estado_abrev, "pais": "MEXICO",
                    "metodo": "abreviatura", "confianza": "ALTA"}

    # ── ETAPA 4: Autopistas / rutas ───────────────────────────────────────
    for ruta, estado_ruta in _SORTED_RUTA:
        if _norm(ruta) in texto:
            return {"estado": estado_ruta, "pais": "MEXICO",
                    "metodo": "autopista", "confianza": "MEDIA"}

    # ── ETAPA 5: Ciudades / municipios ────────────────────────────────────
    mejor_ciudad = None
    mejor_len    = 0
    for ciudad, estado_ciudad in _SORTED_CIUD:
        c_n = _norm(ciudad)
        if c_n in texto and len(c_n) > mejor_len:
            mejor_ciudad = (ciudad, estado_ciudad)
            mejor_len    = len(c_n)
    if mejor_ciudad:
        return {"estado": mejor_ciudad[1], "pais": "MEXICO",
                "metodo": f"ciudad:{mejor_ciudad[0]}", "confianza": "ALTA"}

    # ── ETAPA 6: TADs / TARs / terminales ────────────────────────────────
    for tad_key, estado_tad in _SORTED_TADS:
        if _norm(tad_key) in texto:
            return {"estado": estado_tad, "pais": "MEXICO",
                    "metodo": f"tad:{tad_key}", "confianza": "ALTA"}

    # ── ETAPA 7: Instalaciones PEMEX ──────────────────────────────────────
    for inst_key, estado_inst in _SORTED_INSTAL:
        if _norm(inst_key) in texto:
            return {"estado": estado_inst, "pais": "MEXICO",
                    "metodo": f"pemex:{inst_key}", "confianza": "ALTA"}

    # Señales genéricas PEMEX sin estado específico
    pemex_generic = ["PEMEX", "TAD ", "TAR ", "REFINERIA", "PLATAFORMA",
                     "COMPLEJO PETROQUIMICO", "TERMINAL MARITIMA",
                     "ESTACION DE BOMBEO", "REGION MARINA", "TRAYECTO"]
    for kw in pemex_generic:
        if kw in texto:
            return {"estado": "NO ESPECIFICADO", "pais": "MEXICO",
                    "metodo": f"pemex_generico:{kw.strip()}", "confianza": "MEDIA"}

    # ── ETAPA 8: Fuzzy matching ───────────────────────────────────────────
    # Construir lista de candidatos (ciudades + estados)
    candidatos = {_norm(k): v for k, v in CIUDADES.items()}
    candidatos.update({_norm(k): k for k in ESTADOS_MX})
    match = process.extractOne(
        texto, list(candidatos.keys()),
        scorer=fuzz.partial_ratio, score_cutoff=82
    )
    if match:
        matched_key, score, _ = match
        val = candidatos[matched_key]
        estado_fuzzy = CIUDADES.get(val, ESTADOS_MX.get(val, val))
        if isinstance(estado_fuzzy, str):
            confianza = "ALTA" if score >= 92 else "MEDIA"
            return {"estado": estado_fuzzy, "pais": "MEXICO",
                    "metodo": f"fuzzy:{matched_key}({score}%)", "confianza": confianza}

    return {"estado": "NO ESPECIFICADO", "pais": "NO ESPECIFICADO",
            "metodo": "sin_match", "confianza": "N/A"}


# ─────────────────────────────────────────────────────────────
# FUNCIÓN DE APLICACIÓN AL DATAFRAME
# ─────────────────────────────────────────────────────────────

def agregar_estado(df: pd.DataFrame, col_location: str = "LOCATION") -> pd.DataFrame:
    """
    Aplica clasificar_estado() a todo el DataFrame.
    Agrega / sobreescribe columnas: ESTADO, PAIS, METODO_MATCH, CONFIANZA.
    Usa caché interno para no repetir trabajo en locations idénticas.
    """
    cache: dict = {}
    def _cached(loc):
        k = str(loc).strip().upper()
        if k not in cache:
            cache[k] = clasificar_estado(loc)
        return cache[k]

    resultados = df[col_location].apply(_cached)
    df["ESTADO"]       = resultados.apply(lambda r: r["estado"])
    df["PAIS"]         = resultados.apply(lambda r: r["pais"])
    df["METODO_MATCH"] = resultados.apply(lambda r: r["metodo"])
    df["CONFIANZA"]    = resultados.apply(lambda r: r["confianza"])
    return df


# ─────────────────────────────────────────────────────────────
# APLICAR AL DATAFRAME ACTUAL
# ─────────────────────────────────────────────────────────────

df = agregar_estado(df, col_location="LOCATION")

print("Distribución de PAIS:")
print(df["PAIS"].value_counts())
print("\nDistribución de CONFIANZA:")
print(df["CONFIANZA"].value_counts())


Distribución de PAIS:
PAIS
MEXICO               3150
NO ESPECIFICADO      1104
USA                    23
PANAMA                  9
MAR INTERNACIONAL       3
CHILE                   2
PAISES BAJOS            1
ALEMANIA                1
SINGAPUR                1
Name: count, dtype: int64

Distribución de CONFIANZA:
CONFIANZA
ALTA     3167
N/A      1104
MEDIA      23
Name: count, dtype: int64


### 3.3 Tratamiento de valores duplicados


In [12]:
# ================================
# MANEJO DE LOS DUPLICADOS
# ================================
#region DUPLICADOS
id_col = "CLAIM NUMBER"

def rellenar_informacion_grupo(grupo):
    """
    Completa campos faltantes usando datos de otras filas del mismo CLAIM NUMBER.
    Cubre: LOCATION, DESCRIPTION, ESTADO, PAIS, LoB-Inward, COVER, fechas de poliza.
    """
    VACIOS = ['NO ESPECIFICADO', '', 'nan', 'NAN', 'None']

    def heredar(col, vacios_extra=None):
        if col not in grupo.columns:
            return
        excluir = VACIOS + (vacios_extra or [])
        validos = grupo[
            grupo[col].notna() & ~grupo[col].astype(str).str.strip().isin(excluir)
        ][col].unique()
        if len(validos) > 0:
            mask = grupo[col].isna() | grupo[col].astype(str).str.strip().isin(excluir)
            grupo.loc[mask, col] = validos[0]

    heredar('LOCATION')
    heredar('DESCRIPTION')
    heredar('ESTADO')
    heredar('PAIS')
    heredar('LoB-Inward')
    heredar('COVER')
    heredar('POLICY PERIOD START DATE')
    heredar('POLICY PERIOD END DATE')

    return grupo

def unir_no_especificado(grupo: pd.DataFrame) -> pd.DataFrame:
    """
    Colapsa un grupo de 2 filas del mismo CLAIM NUMBER a 1 sola fila,
    cuando la duplicidad se debe a que una fila quedó con datos
    descriptivos incompletos ("NO ESPECIFICADO") tras el merge/cruce.

    Parámetros
    ----------
    grupo : pd.DataFrame
        Subconjunto de filas de `df` que comparten el mismo CLAIM NUMBER.

    Retorna
    -------
    pd.DataFrame
        - Si el grupo tiene 2 filas, ninguna tiene pagos reales, y una de
          ellas es claramente "la buena" (LOCATION/DESCRIPTION/RESERVE/
          DEDUCTIBLE completos): esa única fila.
        - En cualquier otro caso (incluye: alguna fila SÍ tiene pagos
          reales): el grupo intacto, sin decidir nada — se delega la
          decisión a `seleccionar_mejores_registros`, que sí sabe proteger
          los montos pagados (regla "0️⃣ PAGOS REALES GANAN TODO").
    """
    if len(grupo) != 2:
        return grupo

    # --- Guardia de seguridad: si CUALQUIER fila tiene pagos reales, ---
    # --- no se decide aqui. Se delega a seleccionar_mejores_registros. ---
    cols_pago = ['Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES', 'Cumulative CLAIMS PAID']
    cols_pago_presentes = [c for c in cols_pago if c in grupo.columns]

    if cols_pago_presentes:
        tiene_pago_real = grupo[cols_pago_presentes].fillna(0).gt(0).any(axis=None)
        if tiene_pago_real:
            return grupo

    fila_buena = grupo[
        (grupo["LOCATION"] != "NO ESPECIFICADO") &
        (grupo["GROSS RESERVE"] > 0) &
        (grupo["DEDUCTIBLE"] > 0) &
        (grupo["DESCRIPTION"] != "NO ESPECIFICADO")
    ]
    if len(fila_buena) == 1:
        return fila_buena

    return grupo

num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

def seleccionar_mejores_registros(grupo):
    grupo = grupo.copy()

    # 0️⃣ PAGOS REALES GANAN TODO
    if "Cumulative CLAIMS PAID" in grupo.columns and len(grupo) > 1:
        con_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) > 0]
        sin_paid = grupo[grupo["Cumulative CLAIMS PAID"].fillna(0) == 0]

        if not con_paid.empty and not sin_paid.empty:
            ded_con_paid = set(con_paid["DEDUCTIBLE"].fillna(0).unique())
            sin_paid_con_ded_diff = sin_paid[
                (~sin_paid["DEDUCTIBLE"].fillna(0).isin(ded_con_paid)) &
                (sin_paid["DEDUCTIBLE"].fillna(0) > 0)
            ]
            return pd.concat([con_paid, sin_paid_con_ded_diff])
        elif not con_paid.empty:
            return con_paid

    # 0️⃣.5️⃣ CASO ESPECIAL: STATUS P vs C (Prevalece el registro C)
    if "STATUS" in grupo.columns and len(grupo) > 1:
        fila_p = grupo[grupo["STATUS"] == "P"]
        fila_c = grupo[grupo["STATUS"] == "C"]

        if len(fila_p) == 1 and len(fila_c) == 1:
            cols_gastos = [
                "Cumulative EXPENSES PAID",
                "Cumulative VALUATION EXPENSES",
                "Cumulative CLAIMS PAID",
                "Total Claims Paid Inward",
                "Total Claims Paid no Alae",
                "Inward Incurred",
                "Inward Incurred no Alae"
            ]
            cols_presentes = [c for c in cols_gastos if c in grupo.columns]
            if cols_presentes:
                valores_gastos = fila_p[cols_presentes].fillna(0).iloc[0]
                if (valores_gastos > 0).any():
                    for col in cols_presentes:
                        if fila_c[col].fillna(0).values[0] == 0:
                            grupo.loc[grupo["STATUS"] == "C", col] = valores_gastos[col]
            grupo = grupo[grupo["STATUS"] == "C"]

    # 1️⃣ DEDUCIBLE
    if "DEDUCTIBLE" in grupo.columns and len(grupo) > 1:
        cols_compare = [c for c in grupo.columns if c not in ["DEDUCTIBLE", "STATUS", "STATUS_ORIGINAL", "Nat Cat", "MES CARGA"]]
        dup = grupo.duplicated(subset=cols_compare, keep=False)
        if dup.any():
            candidatos = grupo[dup]
            con_ded = candidatos[candidatos["DEDUCTIBLE"].fillna(0) > 0]
            if not con_ded.empty:
                grupo = con_ded[con_ded["DEDUCTIBLE"] == con_ded["DEDUCTIBLE"].max()]

    # 2️⃣ DEFINIR FINANZAS
    _num_cols = grupo.select_dtypes(include="number").columns
    tiene_finanzas_fila = grupo[_num_cols].fillna(0).sum(axis=1) > 0
    ambos_tienen_finanzas = tiene_finanzas_fila.all()

    # 3️⃣ PAGOS
    cols_pagos = [c for c in ["Cumulative EXPENSES PAID", "Cumulative VALUATION EXPENSES",
                               "Cumulative CLAIMS PAID"] if c in grupo.columns]
    tiene_reserve_o_ded = (
        (grupo["DEDUCTIBLE"].fillna(0) > 0) | (grupo["GROSS RESERVE"].fillna(0) > 0)
    ).any()

    if cols_pagos and len(grupo) > 1 and not tiene_reserve_o_ded:
        tiene_pagos = grupo[cols_pagos].fillna(0).gt(0).any(axis=1)
        if tiene_pagos.any():
            grupo = grupo[tiene_pagos]

    # 4️⃣ ELIMINAR STATUS = C SIN RESERVE NI DED
    if len(grupo) > 1 and ambos_tienen_finanzas:
        eliminar = (
            (grupo["GROSS RESERVE"].fillna(0) == 0) &
            (grupo["DEDUCTIBLE"].fillna(0) == 0) &
            (grupo["STATUS"] == "C")
        )
        if eliminar.any() and (~eliminar).any():
            grupo = grupo[~eliminar]

    # 5️⃣ STATUS P
    if "STATUS" in grupo.columns and len(grupo) > 1 and not ambos_tienen_finanzas:
        status_p = grupo[grupo["STATUS"] == "P"]
        if not status_p.empty:
            grupo = status_p

    # 6️⃣ FILAS CON NÚMEROS
    if len(_num_cols) > 0 and len(grupo) > 1:
        con_numeros = grupo[_num_cols].notna().any(axis=1)
        if con_numeros.any():
            grupo = grupo[con_numeros]

    # 7️⃣ COMPLETITUD
    completitud = grupo.notna().mean(axis=1)
    grupo = grupo.loc[completitud == completitud.max()]

    # 8️⃣ DESEMPATE FINAL
    if len(grupo) > 1 and len(_num_cols) > 0:
        nums = grupo[_num_cols].fillna(0)
        if nums.nunique().max() == 1:
            grupo = grupo.sort_index().tail(1)

    return grupo

# ================================
# APLICAR LOS TRES GROUPBY
# ================================
def procesar_grupo(grupo):
    grupo = rellenar_informacion_grupo(grupo)
    grupo = unir_no_especificado(grupo)
    grupo = seleccionar_mejores_registros(grupo)
    return grupo

df = (
    df.set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(procesar_grupo)
    .reset_index()
)

def auditar_colapso_duplicados(df_antes: pd.DataFrame, df_despues: pd.DataFrame, id_col: str = "CLAIM NUMBER") -> None:
    """
    Verifica que el colapso de filas duplicadas por CLAIM NUMBER no haya
    borrado dinero: compara la suma de columnas monetarias por claim
    antes y despues de `procesar_grupo`, y detiene la ejecucion si algun
    claim perdio monto en Cumulative EXPENSES/VALUATION/CLAIMS PAID.

    Parámetros
    ----------
    df_antes : pd.DataFrame
        Estado de `df` antes de aplicar el groupby de duplicados.
    df_despues : pd.DataFrame
        Estado de `df` despues de aplicar `procesar_grupo`.
    id_col : str, default "CLAIM NUMBER"
        Columna que identifica al siniestro.

    Retorna
    -------
    None
        No retorna nada; imprime un resumen si todo esta bien.

    Excepciones
    -----------
    ValueError
        Si algun CLAIM NUMBER perdio mas de $1 USD en cualquiera de las
        columnas monetarias auditadas entre `df_antes` y `df_despues`.
        Esto detiene el notebook por completo — es una validacion critica
        para no volver a exportar un historico con dinero borrado.
    """
    cols_money = ['Cumulative EXPENSES PAID', 'Cumulative VALUATION EXPENSES', 'Cumulative CLAIMS PAID']
    cols_money = [c for c in cols_money if c in df_antes.columns and c in df_despues.columns]
    if not cols_money:
        print("⚠️ auditar_colapso_duplicados: no se encontraron columnas monetarias para auditar, se omite el chequeo.")
        return

    agg_antes = df_antes.groupby(id_col)[cols_money].sum()
    agg_despues = df_despues.groupby(id_col)[cols_money].sum()
    comp = agg_antes.join(agg_despues, lsuffix='_ANTES', rsuffix='_DESPUES', how='outer').fillna(0)

    perdidas = []
    for col in cols_money:
        diff = comp[f'{col}_DESPUES'] - comp[f'{col}_ANTES']
        afectados = diff[diff < -1]  # tolerancia de $1 por redondeo
        if len(afectados) > 0:
            perdidas.append((col, afectados))

    if perdidas:
        detalle = "\n".join(
            f"  Columna '{col}': {len(afectados)} claim(s) con dinero perdido, total ${afectados.sum():,.2f}\n"
            + afectados.to_string()
            for col, afectados in perdidas
        )
        raise ValueError(
            "\n" + "=" * 60 +
            "\nCOLAPSO DE DUPLICADOS BORRÓ DINERO — EJECUCIÓN DETENIDA\n" +
            "=" * 60 + f"\n{detalle}\n" + "=" * 60 +
            "\nRevisa procesar_grupo / seleccionar_mejores_registros / unir_no_especificado antes de continuar."
        )

    print("✓ Auditoría de colapso de duplicados OK — no se perdió dinero en el proceso.")


# --- Uso: capturar el estado ANTES del groupby, y auditar DESPUES ---
df_antes_dedup = df.copy()

df = (
    df.set_index(id_col)
    .groupby(level=0, group_keys=False)
    .apply(procesar_grupo)
    .reset_index()
)

auditar_colapso_duplicados(df_antes_dedup, df, id_col=id_col)

print(f"✓ {id_col} presente: {id_col in df.columns}")

print(f"✓ {id_col} presente: {id_col in df.columns}")

# ================================
# ✅ PARCHE DE SEGURIDAD: inferir PAIS desde ESTADO
# Cubre casos donde rellenar_informacion_grupo heredó ESTADO
# de otra fila pero PAIS quedó como "NO ESPECIFICADO"
# ================================
if "PAIS" in df.columns and "ESTADO" in df.columns:
    ESTADOS_MEXICO = set(ESTADOS_MX.values())
    mask_sin_pais   = df["PAIS"] == "NO ESPECIFICADO"
    mask_estado_mx  = df["ESTADO"].isin(ESTADOS_MEXICO)
    n_corregidos    = (mask_sin_pais & mask_estado_mx).sum()
    df.loc[mask_sin_pais & mask_estado_mx, "PAIS"] = "MEXICO"
    if n_corregidos > 0:
        print(f"✅ PAIS corregido a MEXICO en {n_corregidos} registros (ESTADO era México pero PAIS era NO ESPECIFICADO)")

# ================================
# ELIMINAR FILAS CON TODOS CEROS
# ================================
num_cols = [
    'Cumulative EXPENSES PAID',
    'Cumulative VALUATION EXPENSES',
    'Cumulative CLAIMS PAID',
    'Total Claims Paid Inward',
    'Total Claims Paid no Alae',
    'Inward Incurred',
    'Inward Incurred no Alae',
    'GROSS RESERVE',
    'DEDUCTIBLE'
]

id_col = "CLAIM NUMBER"

# --- Paso 1: Identificar filas con todos ceros ---
mask_ceros = df[num_cols].fillna(0).sum(axis=1) == 0

# --- Paso 2: Claims con mas de 1 fila ---
filas_por_claim = df.groupby(id_col)[id_col].transform('size')
tiene_multiples_filas = filas_por_claim > 1

# --- Paso 3: Normalizar COVER para comparacion ---
df["_COVER_NORM"] = (
    df["COVER"]
    .fillna("NO ESPECIFICADO")
    .astype(str)
    .str.upper()
    .str.strip()
)

# --- Paso 4: Verificar si la otra fila del claim tiene el mismo COVER ---
def misma_cover_en_grupo(grupo):
    resultado = pd.Series(False, index=grupo.index)
    covers_unicos = grupo["_COVER_NORM"].unique()
    if len(covers_unicos) == 1:
        resultado[:] = True
    else:
        for idx, row in grupo.iterrows():
            cover_actual = row["_COVER_NORM"]
            otras_con_mismo_cover = grupo[(grupo.index != idx) & (grupo["_COVER_NORM"] == cover_actual)]
            if len(otras_con_mismo_cover) > 0:
                resultado[idx] = True
    return resultado

misma_cover = (
    df.groupby(id_col, group_keys=False)
    .apply(misma_cover_en_grupo)
)

# --- Paso 5: Mascara final de eliminacion ---
eliminar = mask_ceros & tiene_multiples_filas & misma_cover

# --- Paso 6: Log de auditoria ---
n_eliminadas = eliminar.sum()
if n_eliminadas > 0:
    print(f"\n{'='*60}")
    print(f"ELIMINACION DE FILAS CON TODOS CEROS")
    print(f"{'='*60}")
    print(f"Filas con todos ceros:           {mask_ceros.sum()}")
    print(f"  - En claims con 1 sola fila:   {(mask_ceros & ~tiene_multiples_filas).sum()} (se conservan)")
    print(f"  - En claims con 2+ filas:      {(mask_ceros & tiene_multiples_filas).sum()}")
    print(f"    - Mismo COVER (eliminables):  {n_eliminadas}")
    print(f"    - COVER diferente (protegidas):{(mask_ceros & tiene_multiples_filas & ~misma_cover).sum()}")
    eliminadas = df[eliminar][[id_col, "COVER", "STATUS"]].copy()
    for _, row in eliminadas.iterrows():
        print(f"  CLAIM {row[id_col]}: COVER={row['COVER']}, STATUS={row['STATUS']}")
else:
    print("No hay filas con ceros candidatas a eliminacion.")

# --- Paso 7: Eliminar y limpiar ---
df = df[~eliminar].copy()
df = df.drop(columns=["_COVER_NORM"])

# --- Paso 8: drop_duplicates CON subset explicito ---
antes_dedup = len(df)
cols_dedup = [c for c in df.columns if c not in ["MES CARGA"]]
df = df.drop_duplicates(subset=cols_dedup)
despues_dedup = len(df)
if antes_dedup != despues_dedup:
    print(f"\nDuplicados exactos eliminados: {antes_dedup - despues_dedup}")

print(f"\nRegistros finales: {len(df)}")

# ================================
# REORDENAR COLUMNAS: CLAIM NUMBER después de CLAIMS-ID
# ================================
cols = df.columns.tolist()
if 'CLAIM NUMBER' in cols and 'CLAIMS-ID' in cols:
    cols.remove('CLAIM NUMBER')
    idx = cols.index('CLAIMS-ID')
    cols.insert(idx + 1, 'CLAIM NUMBER')
    df = df[cols]
    print("✓ CLAIM NUMBER movido después de CLAIMS-ID")

# ================================
# Eliminar columnas CONFIANZA y METODO_MATCH (solo para exportación final, no para análisis interno)
# ================================
df = df.drop(columns=["CONFIANZA", "METODO_MATCH"]) 
print("✓ Columnas CONFIANZA y METODO_MATCH eliminadas para exportación final")    

#endregion

✓ Auditoría de colapso de duplicados OK — no se perdió dinero en el proceso.
✓ CLAIM NUMBER presente: True
✓ CLAIM NUMBER presente: True
No hay filas con ceros candidatas a eliminacion.

Registros finales: 4277
✓ CLAIM NUMBER movido después de CLAIMS-ID
✓ Columnas CONFIANZA y METODO_MATCH eliminadas para exportación final


## 4. Exportación de resultados


In [ ]:
# Guardar Excel
# Definición de rutas (Asegúrate de que las variables estén declaradas previamente)
ruta_salida2 = os.path.join(carpeta_salida2, nombre_archivo_salida)
ruta_salida = os.path.join(carpeta_salida, nombre_archivo_salida)

# ================================
# COLUMNAS A EXCLUIR DE LA EXPORTACION
# ================================
import re
# Detecta dinamicamente cualquier columna que termine en espacio + 6 digitos (YYYYMM)
# Ejemplo: 'GROSS RESERVE 202504', 'Cumulative CLAIMS PAID 202510', etc.
cols_con_periodo = [c for c in df.columns if re.search(r' \d{6}$', c)]

# Columnas fijas a excluir siempre
cols_fijas = [
    'CLAIM NUMBER ORIGINAL BDX', 'COVER BDX',
    'POLICY PERIOD START DATE BDX', 'POLICY PERIOD END DATE BDX',
    'Monthly', 'CEDANT OBSERVATIONS', 'KOT OBSERVATIONS',
    'BDX_SOURCE_FILE', 'BDX_SOURCE_SHEET', 'COVER_RAW', 'DIF OSLR','KEY DED','FLAG CONTA'
]
cols_a_dropear = [c for c in cols_con_periodo + cols_fijas if c in df.columns]
df_export = df.drop(columns=cols_a_dropear)
print(f'Columnas con periodo detectadas: {cols_con_periodo}')
print(f'Total columnas eliminadas de la exportacion: {len(cols_a_dropear)}')

# ================================
# VALIDACIÓN PRE-EXPORTACIÓN
# ================================
print(f"\n=== VALIDACIÓN FINAL ===")
print(f"Total registros: {len(df)}")

# Evitar KeyError si la columna no existe aún en el flujo
if "CLAIM NUMBER" in df.columns:
    print(f"Claims únicos: {df['CLAIM NUMBER'].nunique()}")
else:
    print("ALERTA: La columna 'CLAIM NUMBER' no existe en el DataFrame.")

print(f"\nDistribución STATUS:")
print(df["STATUS"].value_counts())

print(f"\nDistribución COVER:")
print(df["COVER"].value_counts())

print(f"\nDistribución LOB-INWARD:")
print(df["LoB-Inward"].value_counts())

# Corregido: Apunta a STATUS que es la columna homologada
registros_sin_status = df["STATUS"].isin(["NO ESPECIFICADO", "NAN"]).sum()
print(f"\nRegistros sin STATUS (No especificado): {registros_sin_status}")

# Verificar NaN o valores "vacíos de negocio" en columnas clave
columnas_clave = ["STATUS", "COVER"]
if "CLAIM NUMBER" in df.columns:
    columnas_clave.append("CLAIM NUMBER")

for col in columnas_clave:
    # Cuenta nulos reales de Pandas
    nulos_reales = df[col].isna().sum()
    # Cuenta nulos simulados por la conversión a string
    nulos_texto = df[col].astype(str).str.upper().isin(["NAN", "NO ESPECIFICADO", ""]).sum()
    
    if nulos_reales > 0 or nulos_texto > 0:
        print(f"⚠️ ALERTA: '{col}' tiene {nulos_reales} NaN reales y {nulos_texto} valores vacíos/sin especificar.")

#Redondear columnas numéricas a 2 decimales
df_export = df_export.round(2)
# ================================
# EXPORTACIÓN A EXCEL (xlsxwriter)
# ================================
with pd.ExcelWriter(ruta_salida2, engine="xlsxwriter") as writer:
    df_export.to_excel(writer, sheet_name="Hoja1", index=False)
    
    workbook  = writer.book

with pd.ExcelWriter(ruta_salida, engine="xlsxwriter") as writer:
    df_export.to_excel(writer, sheet_name="Hoja1", index=False)
    
    workbook  = writer.book

Columnas con periodo detectadas: []
Total columnas eliminadas de la exportacion: 1

=== VALIDACIÓN FINAL ===
Total registros: 4277
Claims únicos: 4269

Distribución STATUS:
STATUS
P    2709
C    1389
T     179
Name: count, dtype: int64

Distribución COVER:
COVER
CARGO                              3371
CARGA POLIETILENO                   359
RC FLETADORES(PMI)                  200
P&I                                 145
CASCO Y MAQ.                        121
RC FLETADORES(PEMEX)                 74
JACK-UPS(DAÑO FISICO)                 2
DEEP WATERS                           2
NO ESPECIFICADO                       2
EQUIPO FERROVIARIO(DAÑO FÍSICO)       1
Name: count, dtype: int64

Distribución LOB-INWARD:
LoB-Inward
CARGO                              3730
P&I                                 419
CASCO Y MAQ.                        121
JACK-UPS(DAÑO FISICO)                 2
DEEP WATERS                           2
NO ESPECIFICADO                       2
EQUIPO FERROVIARIO(DAÑO FÍSICO)   

C:\Users\IKAL14\AppData\Local\Temp\ipykernel_188956\2565159170.py:66: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  df_export = df_export.round(2)


: 